# 10. Futility-анализ


## План занятия

0. **Постановка задачи** — efficacy vs futility
1. **Математическая разработка подхода**
    1. **$Z$ и $B$ значения**
    2. **Conditional Power (CP)**
    3. **Predictive Power (PPoS)**
    4. **$\beta$-spending**
        1. Предварительная фиксация futility-границы
        2. **Binding** и **non-binding** фиксация
2. **Влияние futility на ошибки I и II рода**
3. **Сравнение всех методов**
4. **Практические рекомендации**
5. **Итоги**


## Библиотеки

In [25]:
import math
from dataclasses import dataclass
from tqdm import tqdm
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from scipy.stats import norm
from scipy.optimize import brentq
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

pio.templates.default = "plotly_white"

SEED = 12345
rng = np.random.default_rng(SEED)

# Единая палитра исходов на всю лекцию
C_FUT = "#d95f02"   # оранжевый — futility
C_EFF = "#1b9e77"   # зелёный — efficacy
C_CONT = "#999999"  # серый — продолжаем


def style(fig, height=440, title=None, xlabel=None, ylabel=None):
    """Единый стиль всех графиков: сетка, легенда, подписи."""
    fig.update_layout(
        template="plotly_white",
        height=height,
        title=dict(text=title, font=dict(size=15)),
        xaxis_title=xlabel, yaxis_title=ylabel,
        margin=dict(l=60, r=20, t=70, b=50),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0, font=dict(size=11)),
    )
    fig.update_xaxes(showgrid=True, gridcolor="rgba(0,0,0,0.07)", zeroline=True)
    fig.update_yaxes(showgrid=True, gridcolor="rgba(0,0,0,0.07)", zeroline=True)
    return fig



In [3]:
@dataclass(frozen=True)
class Design:
    """Параметры теста непрерывной метрики (средний чек)."""
    alpha: float = 0.05        # односторонняя ошибка I рода
    beta: float = 0.20         # ошибка II рода (power = 0.80)
    sigma: float = 100.0       # СКО метрики, руб.
    mde: float = 10.0          # абсолютный MDE, руб.
    K: int = 10
    t: tuple = (0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0)  # доли информации (моменты взглядов)

    @property
    def delta(self) -> float:
        return self.mde / self.sigma

    @property
    def theta_mde(self) -> float:
        return norm.ppf(1 - self.alpha) + norm.ppf(1 - self.beta)

    @property
    def i_max(self) -> float:
        return (self.theta_mde / self.delta) ** 2

    @property
    def n_max(self) -> float:
        return 2 * self.sigma ** 2 * self.i_max

design = Design()
design

Design(alpha=0.05, beta=0.2, sigma=100.0, mde=10.0, K=10, t=(0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0))

## 0. Постановка задачи


Есть две разновидности ранней остановки эксперимента:

- **Остановка по эффективности (efficacy)** — эффект статистически доказан до окончания эксперимента. Останавливаемся, потому что *победили*.
- **Остановка по бесперспективности (futility)** — эффекта, скорее всего, нет или он меньше MDE. Останавливаемся, чтобы не жечь трафик на заведомо «мёртвый» тест.

**Остановка по бесперспективности (futility)** появляется как естественный ответ на следующий вопрос:
> *Если на промежуточном взгляде эффект слабее MDE (или его нет вовсе), с какой вероятностью тест всё-таки дойдёт до значимости к концу?*

Если эта вероятность мала — тратить оставшийся трафик бессмысленно, и можно завершить тест.

## 1. Математическая разработка подхода

### 1.1. $Z$ и $B$ значения

<details>
<summary>Вывод распределения $Z_t$</summary>

Пусть у нас есть группы $A$ (контроль) и $B$ (тест), по $n_t$ i.i.d. наблюдений в каждой с общей дисперсией $\sigma^2$. В качестве эффекта мы будем оценивать разность средних: $\hat\delta_t=\bar X_B-\bar X_A$.

Дисперсия оцениваемой разницы среднего будет $\mathrm{Var}(\hat\delta_t)=2\sigma^2/n_t$, а $SE_t=\sigma\sqrt{2/n_t}$
1. $\mathrm{Var}(\bar X_A)=\mathrm{Var}\!\left(\frac1{n_t}\sum X_{A,i}\right)=\frac{1}{n_t^2}\sum\mathrm{Var}(X_{A,i})=\frac{1}{n_t^2}\cdot n_t\sigma^2=\frac{\sigma^2}{n_t}$.
2. Аналогично п.1 для $B$.
3. Так как выборки независимы, дисперсии складываются: $\mathrm{Var}(\hat\delta_t)= \mathrm{Var}(\bar X_A) + \mathrm{Var}(\bar X_B)$ = $2\sigma^2/n_t$
4. $SE_t=\sqrt{\mathrm{Var}(\hat\delta_t)} = \sigma\sqrt{2/n_t}$.

После стандартизации $\hat\delta_t$ получаем величину $Z_t :=\hat\delta_t/SE_t$, которая распределена как $\mathcal N\big(\delta/SE_t,\,1\big)$.

Введём **информацию** $I_t$ - величину обратную дисперсии оценки; $I_t =1/SE_t^2=n_t/(2\sigma^2)$. Через $I_t$ обозначим момент времени $t$, который показывает какую часть информации мы увидели: $t=I_t/I_{max}\approx n_t/n_{max}$.

Через информацию также обозначим и **дрейф** $\theta$ - наблюдаемый эффект, отмасштабированный на дисперсию: $\theta=\delta\sqrt{I_{max}}$.


Через эти 2 новые величины переобозначим математическое ожидание $Z_t$  для более компактной записи, которой будем придерживаться дальше все время:

$$\mathbb E[Z_t]=\frac{\delta}{SE_t}=\delta\sqrt{I_t}=\underbrace{\delta\sqrt{I_{max}}}_{=:\theta}\cdot\underbrace{\sqrt{I_t/I_{max}}}_{=\sqrt t}=\theta\sqrt t.$$

$$\boxed{Z_t\sim\mathcal N(\theta\sqrt t,\,1)}$$

</details>

<details>
<summary>Независимые приращения и ковариация</summary>

<details>
<summary>Факт 1. Оценка в момент $t$ раскладывается на старый и новый куски $$Z_t=\sqrt{\frac{s}{t}} \cdot Z_s+\sqrt{\frac{t-s}{t}} \cdot Z_{t-s}.$$</summary>

  Новые данные не знают о старых. Данные приходят порциями, и оценка в момент $t$ — это взвешенное среднее старой оценки и оценки только по новым данным. Распишем среднее по $n_t$ наблюдениям как сумму по первым $n_s$ и по остальным $n_t-n_s$:

$$\bar X_B^{(t)}=\frac1{n_t}\sum_{i=1}^{n_t}X_{B,i}=\frac1{n_t}\left(\sum_{i=1}^{n_s}X_{B,i}+\sum_{i=n_s+1}^{n_t}X_{B,i}\right)=\frac{n_s}{n_t}\bar X_B^{(s)}+\frac{n_t-n_s}{n_t}\bar X_B^{(t-s)}.$$

То же для $A$, вычитаем:

$$\hat\delta_t=\frac{n_s}{n_t}\hat\delta_s+\frac{n_t-n_s}{n_t}\hat\delta_{t-s}.$$

Делим обе части на $SE_t$:

$$Z_t=\frac{n_s}{n_t}\cdot\frac{\hat\delta_s}{SE_t}+\frac{n_t-n_s}{n_t}\cdot\frac{\hat\delta_{t-s}}{SE_t}.$$

Каждый кусок пересчитывается в свою нормированную статистику:

$$
\frac{\hat\delta_s}{SE_t}=\frac{\hat\delta_s}{SE_s}\cdot\frac{SE_s}{SE_t}=Z_s\cdot\frac{SE_s}{SE_t} = Z_s\cdot\frac{\sigma\sqrt{2/n_s}}{\sigma\sqrt{2/n_t}}=Z_s\cdot\sqrt{\frac{n_t}{n_s}} \\
\frac{\hat\delta_{t-s}}{SE_t}=\frac{\hat\delta_{t-s}}{SE_{t-s}}\cdot\frac{SE_{t-s}}{SE_t}=Z_{t-s}\cdot\frac{SE_{t-s}}{SE_t} = Z_{t-s}\cdot\frac{\sigma\sqrt{2/n_{t-s}}}{\sigma\sqrt{2/n_t}}=Z_{t-s}\cdot\sqrt{\frac{n_t}{n_{t-s}}}=Z_{t-s}\cdot\sqrt{\frac{n_t}{n_t-n_s}}
$$

Тогда

$$Z_t=\frac{n_s}{n_t} \cdot \sqrt{\frac{n_t}{n_s}} \cdot Z_s + \frac{n_t-n_s}{n_t} \cdot \sqrt{\frac{n_t}{n_t-n_s}} \cdot Z_{t-s} \\
Z_t=\sqrt{\frac{n_s}{n_t}} \cdot Z_s+\sqrt{\frac{n_{t-s}}{n_t}} \cdot Z_{t-s}=\sqrt{\frac{s}{t}} \cdot Z_s+\sqrt{\frac{t-s}{t}} \cdot Z_{t-s}
$$    
</details>

<details>
<summary>Факт 2. Ковариация и корреляция $Z$-величин двух периодов пропорционально корню отношения моментов $$\mathrm{Cov}(Z_s,Z_t)=\mathrm{corr}(Z_s,Z_t)=\sqrt{s/t}$$</summary>

Для доказательства используем билинейность ковариации:
$$\mathrm{Cov}(aX+bY,Z)=a \cdot \mathrm{Cov}(X,Z)+b \cdot \mathrm{Cov}(Y,Z)$$

Берём разложение $Z_t=\sqrt{\frac{s}{t}} \cdot Z_s+\sqrt{\frac{t-s}{t}} \cdot Z_{t-s}$, и подставим в формулу:

$$\mathrm{Cov}(Z_s,Z_t)=\mathrm{Cov}\Big(Z_s, \sqrt{\frac{s}{t}} \cdot Z_s+\sqrt{\frac{t-s}{t}} \cdot Z_{t-s}\Big)$$

По билинейности разваливается на два слагаемых:

$$=\sqrt{\frac{s}{t}} \cdot \underbrace{\mathrm{Cov}(Z_s,Z_s)}_{=1} + \sqrt{\frac{t-s}{t}} \cdot \underbrace{\mathrm{Cov}(Z_{t-s},Z_s)}_{=0}=\sqrt{\frac{s}{t}}.$$

Так как $\mathrm{Var}(Z_s)=\mathrm{Var}(Z_t)=1$, то и
$$\mathrm{corr}(Z_s,Z_t)=\sqrt{s/t}$$
</details>
</details>

<details>
<summary>Переход к $B(t)$</summary>

$Z_t$ прекрасен тем, что безразмерен и всегда $\mathcal N(\theta\sqrt t,1)$. Но у него есть две неприятности, которые мешают строить границы.

**Дефект 1**: $Z$ не «накапливается». Дисперсия всегда 1, значит прирост дисперсии нулевой:

$$\mathrm{Var}(Z_t)-\mathrm{Var}(Z_s)=1-1=0.$$

А само приращение $Z_t-Z_s$ имеет дисперсию

$$\mathrm{Var}(Z_t-Z_s)=1+1-2\sqrt{s/t},$$

которая зависит от обоих моментов и не равна длине интервала, т.е. «сколько нового пришло» в единицах $Z$ не измеряется.

**Дефект 2**: в разложении появляются веса при $Z_s$ и $Z_{t-s}$, которые надо таскать.

<br>

Обе проблемы — от одного корня: мы поделили на $SE_t$, а $SE_t\propto1/\sqrt t$ меняется со временем. Чтобы избавиться от переменного делителя домножим на $\sqrt t$  и будем вести расчёты в новой величине $B_t= \sqrt t \cdot Z_t = \sqrt t\cdot\frac{\hat\delta_t}{SE_t}$.
У новой величины получаются следующие моменты и величины:
1. $\mathbb E[B(t)]=\sqrt t\cdot\theta\sqrt t=\theta t$
2. $\mathrm{Var}[B(t)]=t\cdot1=t$
3. $\mathrm{Cov}\big(B(s),B(t)\big)=\mathrm{Cov}\big(\sqrt s \cdot Z_s,\sqrt t \cdot Z_t\big)=\sqrt{st}\cdot\underbrace{\mathrm{Cov}(Z_s,Z_t)}_{=\sqrt{s/t}}=\sqrt{st}\cdot\sqrt{\frac{s}{t}}=s$
4. $\mathrm{Cov}\big(B(s),B(t)-B(s)\big)=\mathrm{Cov}(B(s),B(t))-\mathrm{Var}(B(s))=s-s=0$

<br>

На шкале $B$ разложение — простое сложение без весов:

$$B(t)=\sqrt t\left(\sqrt{\frac{s}{t}}Z_s+\sqrt{\frac{t-s}{t}}Z_{t-s}\right)=\sqrt s \cdot Z_s+\sqrt{t-s} \cdot Z_{t-s}=B(s)+\Delta B$$


Так как $(B(s),B(t))$ совместно нормальны (это линейные комбинации независимых нормальных приращений), нулевая ковариация означает независимость:

$$B(t)-B(s)\perp B(s) , \qquad
B(t)-B(s)\sim\mathcal N(\theta(t-s),t-s)$$

<br>

Величина $B$ так же имеет и содержательное ядро:

Подставим $Z_t=\hat\delta_t\sqrt{I_t}$ (так как $SE_t=1/\sqrt{I_t}$):

$$B(t)=\sqrt t\,Z_t=\sqrt{\frac{I_t}{I_{max}}}\cdot\hat\delta_t\sqrt{I_t}=\frac{\hat\delta_t I_t}{\sqrt{I_{max}}}.$$

Раскроем $\hat\delta_t I_t$ через данные:

$$\hat\delta_t I_t=\left(\frac1{n_t}\sum_{i=1}^{n_t}\Delta_i\right)\frac{n_t}{2\sigma^2}=\frac{1}{2\sigma^2}\sum_{i=1}^{n_t}\Delta_i,\qquad \Delta_i=X_{B,i}-X_{A,i}.$$

Итого

$$\boxed{B(t)=\frac{1}{2\sigma^2\sqrt{I_{max}}}\sum_{i=1}^{n_t}\big(X_{B,i}-X_{A,i}\big)}$$

$B$ — с точностью до константы простая накопленная сумма по пользователям. Получается, что прирост $B(t)-B(s)$ - сумма по новым пользователям, независимая от суммы по старым. Формально $B$ — броуновское движение с дрейфом $\theta$.
<br>

**Переход границ между шкалами.** Efficacy-границу на взгляде $k$ в $Z$-шкале обозначим $e_k$, futility-границу — $f_k$. Правило на $Z$-шкале $Z_k>e_k$ домножением на $\sqrt{t_k}$ превращается в

$$Z_k>e_k\iff B(t_k)>e_k\sqrt{t_k},\qquad Z_k<f_k\iff B(t_k)<f_k\sqrt{t_k}.$$

Вся дальнейшая техника работает на шкале $B$, где границы — это $e_k\sqrt{t_k}$ и $f_k\sqrt{t_k}$, а результаты возвращаются на $Z$ делением на $\sqrt{t_k}$ (получаем $e_k$ и $f_k$).
</details>

In [4]:
def z_stat(xA: np.ndarray, xB: np.ndarray, sigma: float) -> float:
    """Двухвыборочный z-тест непрерывной метрики (известная дисперсия)."""
    nA, nB = xA.size, xB.size
    se = sigma * np.sqrt(1 / nA + 1 / nB)
    return (xB.mean() - xA.mean()) / se


def simulate_b(rng, theta: float, t: np.ndarray, n_sim: int = 200) -> np.ndarray:
    """Траектории B(t) = sqrt(t)*Z_t — броуновское движение с дрейфом theta."""
    dt = np.diff(np.concatenate(([0.0], t)))
    inc = rng.normal(loc=theta * dt, scale=np.sqrt(dt), size=(n_sim, len(t)))
    return np.cumsum(inc, axis=1)


In [5]:
t = np.asarray(design.t)

modes = [
    (0.0,              "H<sub>0</sub> (θ = 0)",    C_CONT),
    (design.theta_mde, "H<sub>1</sub> (θ = MDE)",  C_EFF),
]

B_by_mode = [(theta, name, color, simulate_b(rng, theta, t, 250))
             for theta, name, color in modes]

x_ticks = np.linspace(t.min(), t.max(), 20)
y_all   = np.concatenate([B.ravel() for *_, B in B_by_mode])
y_lo    = min(y_all.min(), 0.0)
y_hi    = max(y_all.max(), modes[1][0] * t.max())
y_ticks = np.linspace(y_lo, y_hi, 20)

# Подписи панелей — с расшифровкой смысла, чтобы легенда была не нужна
subplot_titles = (
    "H<sub>0</sub> (θ = 0)",
    "H<sub>1</sub> (θ = MDE)",
)

fig = make_subplots(
    rows=1, cols=2, shared_yaxes=True,
    horizontal_spacing=0.06,
    subplot_titles=subplot_titles,
)

for col, (theta, name, color, B) in enumerate(B_by_mode, start=1):
    for i in range(B.shape[0]):
        fig.add_trace(
            go.Scatter(
                x=t, y=B[i], mode="lines",
                line=dict(color=color, width=0.8),
                opacity=0.25,
                showlegend=False,
                hoverinfo="skip",
            ),
            row=1, col=col,
        )
    # Чёрная штриховая линия дрейфа (легенды нет — подпишем аннотацией ниже)
    fig.add_trace(
        go.Scatter(
            x=t, y=theta * t, mode="lines",
            line=dict(color="black", width=2.5, dash="dash"),
            showlegend=False,
            name=f"E[<i>B</i>(<i>t</i>)] = {theta:.2f}·<i>t</i>",
        ),
        row=1, col=col,
    )

    # Небольшая подпись у линии дрейфа — вместо легенды
    fig.add_annotation(
        text=f"E[<i>B</i>(<i>t</i>)] = {theta:.2f}·<i>t</i>",
        x=t.max(), y=theta * t.max(),
        xanchor="right", yanchor="bottom",
        showarrow=False,
        font=dict(size=11, color="black"),
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="rgba(0,0,0,0.35)", borderwidth=1,
        row=1, col=col,
    )

fig.update_xaxes(
    tickmode="array", tickvals=x_ticks, tickformat=".2f",
    title_text="Доля информации <i>t</i>", automargin=True,
)
fig.update_yaxes(
    tickmode="array", tickvals=y_ticks, tickformat=".2f",
    automargin=True,
)
fig.update_yaxes(title_text="<i>B</i>(<i>t</i>)", col=1)

fig.update_layout(
    autosize=True,
    height=720,
    margin=dict(t=70, b=80, l=70, r=40),
    title=dict(
        text="Броуновское движение при H<sub>0</sub> и H<sub>1</sub>",
        x=0.5, xanchor="center",
        y=0.98, yanchor="top", yref="container",
        font=dict(size=16),
    ),
    showlegend=False,
)

fig

### 1.2. Conditional Power (CP)

<details>
<summary>Условная вероятность «добить» тест</summary>

$\mathrm{CP}(\theta)=P_\theta(Z_1>z_{1-\alpha}\mid Z_t=z_t)$ — вероятность дойти до значимости при заданном дрейфе $\theta$ и текущих данных. Это и есть ответ на вопрос «стоит ли тратить оставшийся трафик».


Выразим финальную статистику через текущую и приращение:
$$Z_1=B(1)=B(t)+\big(B(1)-B(t)\big)=\sqrt t \cdot z_t+\big(B(1)-B(t)\big).$$

Приращение независимо от $B(t)$ и распределено как $\mathcal N(\theta(1-t),1-t)$. Поэтому при условии $Z_t=z_t$ первое слагаемое — детерминированное число $\sqrt t \cdot z_t$, а второе — случайное. Получается, что финальная статистика имеет условное распределение:
$$(Z_1 \mid Z_t=z_t) \sim \mathcal N\big(\sqrt t \cdot z_t+\theta(1-t);1-t\big)$$

<br>
Вероятность попасть в критическую область $\{Z_1>z_{1-\alpha}\}$ — хвост этого распределения.
Стандартизуем: $\mu=\sqrt t z_t+\theta(1-t)$, $\sigma=\sqrt{1-t}$:
$$\mathrm{CP}(\theta)=P \left(\frac{Z_1-\mu}{\sigma}>\frac{z_{1-\alpha}-\mu}{\sigma}\right)=1-\Phi\left(\frac{z_{1-\alpha}-\sqrt t \cdot z_t-\theta(1-t)}{\sqrt{1-t}}\right)$$

$$\boxed{\mathrm{CP}(\theta)=1-\Phi\left(\frac{z_{1-\alpha}-\sqrt t \cdot z_t-\theta(1-t)}{\sqrt{1-t}}\right)}$$


Предельные случаи.

* $t\to0$: $\sqrt t z_t\to0$, $\theta(1-t)\to\theta$, $\sqrt{1-t}\to1$, поэтому $\mathrm{CP}\to1-\Phi(z_{1-\alpha}-\theta)$ — априорная мощность дизайна.
* $t\to1$: знаменатель $\sqrt{1-t}\to0$. При $z_t<z_{1-\alpha}$ числитель $\to z_{1-\alpha}-z_t>0$, то есть $u\to+\infty$, $\Phi\to1$, $\mathrm{CP}\to0$. При $z_t>z_{1-\alpha}$ числитель $\to$ отрицателен, $\mathrm{CP}\to1$. Получается ступенька в точке $z_t=z_{1-\alpha}$.
Выбор $\theta$ — см. блок «Какое $\theta$ подставить» ниже.
</details>

<details>
<summary>Монотонность по $z_t$</summary>

Обозначим $u=(z_{1-\alpha}-\sqrt t\,z_t-\theta(1-t))/\sqrt{1-t}$. Тогда

$$\frac{d\,\mathrm{CP}}{dz_t}=-\Phi'(u)\cdot\frac{du}{dz_t}=-\varphi(u)\cdot\left(-\frac{\sqrt t}{\sqrt{1-t}}\right)=\varphi(u)\sqrt{\frac{t}{1-t}}>0.$$

Поскольку плотность $\varphi(u)>0$, CP строго возрастает по $z_t$ — значит, уравнение $\mathrm{CP}=\gamma$ имеет единственный корень.
</details>


<details>
<summary>Какое $\theta$ подставить</summary>
Какое $\theta$ подставить — самый тонкий момент. Три осмысленных варианта:

1. $\theta=0$ (под $H_0$): «даже если эффекта нет, дойдём ли до ложной значимости?» — консервативно, редко останавливает.
2. $\theta=\hat\theta_t=z_t/\sqrt t$ (текущий тренд): «будущее похоже на то, что уже видим» — заманчиво, но оптимистично: ранняя оценка шумная, и случайно «выстреливший» эффект кажется многообещающим.
3. $\theta=\theta_{\mathrm{MDE}}$ (дизайновый эффект): «если эффект ровно такой, как закладывали в power-расчёте» — стандартный ориентир, отвечает на вопрос «есть ли шанс поймать то, ради чего затевался тест».
</details>


Правило futility: **если $\mathrm{CP} < \gamma$ — останавливаемся** (типично $\gamma \in [0.10, 0.20]$).


#### 1.2.1. Построение CP

In [6]:
def conditional_power(z_t, t, theta, z_alpha=1.6448536):
    """CP: вероятность Z_1 > z_alpha при наблюдаемом z_t и истинном дрейфе theta.

    Порог futility по CP (f_k^CP) — значение z_t, при котором CP = gamma.
    """
    z_t = np.asarray(z_t, dtype=float)
    return 1.0 - norm.cdf((z_alpha - np.sqrt(t) * z_t - theta * (1 - t)) / np.sqrt(1 - t))


def ppos(z_t, t, theta0, tau, z_alpha=1.6448536):
    """Predictive probability of success (нормальный prior, замкнутая форма)."""
    z_t = np.asarray(z_t, dtype=float)
    sigma2 = 1.0 / (1.0 / tau**2 + t)
    mu = sigma2 * (theta0 / tau**2 + np.sqrt(t) * z_t)
    return 1.0 - norm.cdf((z_alpha - np.sqrt(t) * z_t - mu * (1 - t))
                          / np.sqrt((1 - t) + (1 - t)**2 * sigma2))


In [7]:
z_alpha = norm.ppf(1 - design.alpha)
z_grid = np.linspace(-3, 3, 300)
tt = 0.5

fig = go.Figure()
fig.add_trace(go.Scatter(x=z_grid, y=conditional_power(z_grid, tt, 0.0, z_alpha),
                         mode="lines", line=dict(color=C_CONT, width=2.5), name="CP под H<sub>0</sub>"))
fig.add_trace(go.Scatter(x=z_grid, y=conditional_power(z_grid, tt, design.theta_mde, z_alpha),
                         mode="lines", line=dict(color=C_EFF, width=2.5), name="CP под θ<sub>MDE</sub>"))
fig.add_trace(go.Scatter(x=z_grid, y=conditional_power(z_grid, tt, z_grid / np.sqrt(tt), z_alpha),
                         mode="lines", line=dict(color=C_FUT, width=2.5), name="CP под трендом"))
fig.add_hline(y=0.15, line_dash="dash", line_color="black", annotation_text="порог γ = 0.15")
fig = style(fig, title=f"CP при <i>t</i> = {tt}: выбор θ кардинально меняет ответ",
            xlabel="наблюдаемая <i>z</i><sub>t</sub>", ylabel="Conditional Power")
fig


#### 1.2.2. Карта CP

Порог $\mathrm{CP}<\gamma$ перепишем в терминах статистики. При фиксированных $t,\alpha,\gamma,\theta$ определим $f^{\mathrm{CP}}$ как решение

$$\mathrm{CP}\big(f^{\mathrm{CP}};t,\theta\big)=\gamma.$$

Так как CP монотонно растёт по $z_t$ (показано в §1.2, блок «Монотонность по $z_t$»), это уравнение имеет единственный корень, и

$$\mathrm{CP}(z_t)<\gamma\iff z_t<f^{\mathrm{CP}}(t).$$

Численно корень ищут методом деления пополам или Брента (в лекции — `brentq`). На плоскости $(t,z)$ правило «стоп по CP» — кривая $f^{\mathrm{CP}}(t)$, ниже которой останавливаемся.

Карта CP на плоскости $(t, z)$ показывает, почему время взгляда так важно: на ранних взглядах карта почти плоская (судьбу решает будущее), к концу «заостряется» в ступеньку. Красный контур $\mathrm{CP}=\gamma$ — граница «стоп / продолжаем».

In [8]:
from functools import lru_cache
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm
from scipy.optimize import brentq
from ipywidgets import interact, FloatSlider

Z_ALPHA = 1.6448536


@lru_cache(maxsize=4096)
def _cp_boundary_cached(t_round, gamma, theta, z_alpha=Z_ALPHA):
    f = lambda z: conditional_power(z, t_round, theta, z_alpha) - gamma
    try:
        return brentq(f, -8, 8)
    except ValueError:
        return np.nan


def cp_to_boundary(t, gamma, theta, z_alpha=Z_ALPHA):
    return _cp_boundary_cached(round(float(t), 4), float(gamma), float(theta), z_alpha)


def plot_cp_heatmap(theta=design.theta_mde, gamma=0.15):
    t_grid = np.linspace(0.05, 0.95, 120)
    z_grid = np.linspace(-5, 5, 240)
    T, Z = np.meshgrid(t_grid, z_grid)
    CP = conditional_power(Z, T, theta, Z_ALPHA)

    f_cp = np.array([cp_to_boundary(t_, gamma, theta, Z_ALPHA) for t_ in t_grid])

    fig = go.Figure()
    fig.add_trace(go.Heatmap(
        z=CP, x=t_grid, y=z_grid,
        colorscale="Viridis",
        colorbar=dict(title="CP"),
        showscale=True,
        zmin=0, zmax=1,
    ))
    fig.add_trace(go.Scatter(
        x=t_grid, y=f_cp, mode="lines",
        line=dict(color="red", width=3),
        name=f"<i>f</i><sup>CP</sup>: CP = {gamma:.2f}",
    ))

    fig.update_layout(
        autosize=True,
        height=520,
        margin=dict(l=70, r=20, t=70, b=60),
        uirevision="cp_heatmap",
        # HTML-разметка вместо MathJax — рендерится всегда
        title=dict(
            text=(f"Conditional power под MDE: контур — граница futility "
                  f"<i>f</i><sup>CP</sup> (θ = {theta:.2f}, γ = {gamma:.2f})"),
            x=0.5, xanchor="center",
        ),
        xaxis=dict(
            title="Доля информации <i>t</i>",
            dtick=0.05,
            tickformat=".2f",
            showgrid=True,
            zeroline=False,
        ),
        yaxis=dict(
            title="<i>z</i><sub>t</sub>",
            dtick=1,
            tickformat=".0f",
            showgrid=True,
            zeroline=False,
        ),
    )
    fig.update_xaxes(range=[t_grid[0], t_grid[-1]])
    fig.update_yaxes(range=[z_grid[0], z_grid[-1]])

    fw = go.FigureWidget(fig)
    fw.layout.autosize = True
    return fw


interact(
    plot_cp_heatmap,
    theta=FloatSlider(value=design.theta_mde, min=0, max=4, step=0.1,
                      description="θ"),
    gamma=FloatSlider(value=0.15, min=0.05, max=0.50, step=0.05,
                      description="γ"),
)

interactive(children=(FloatSlider(value=2.4864748605243863, description='θ', max=4.0), FloatSlider(value=0.15,…

<function __main__.plot_cp_heatmap(theta=np.float64(2.4864748605243863), gamma=0.15)>

#### 1.2.3. Особенность CP

CP-граница строится из условного расчёта на каждом взгляде (подставляем наблюдаемое $z_t$) и интерпретируется как «вероятность успеха», но она не гарантирует заранее, сколько ошибки II рода будет «съедено» по всем траекториям. Альтернатива — зафиксировать границу до данных с гарантией контроля ошибки II рода: $\beta$-spending (§1.4).

### 1.3. Predictive Power (PPoS)
<details>
<summary>Определение</summary>

$$\mathrm{PPoS}=\int \mathrm{CP}(\theta)\,\pi(\theta\mid D_t)\,d\theta.$$

Вместо «шансы при одном фиксированном $\theta$» — «средние шансы по всем правдоподобным $\theta$, взвешенным апостериорной вероятностью»

</details>

<details>
<summary>Точность апостериора $\pi(\theta)$ </summary>

Априор $\theta\sim\mathcal N(\theta_0,\tau^2)$. Правдоподобие $Z_t\mid\theta\sim\mathcal N(\theta\sqrt t,1)$ эквивалентно наблюдению $y=z_t/\sqrt t$ с шумом $y\mid\theta\sim\mathcal N(\theta,1/t)$. Плотности:

$$\pi(\theta)\propto\exp\left(-\frac{(\theta-\theta_0)^2}{2\tau^2}\right),\qquad L(\theta)\propto\exp\left(-\frac{t(\theta-y)^2}{2}\right)$$

Апостериор $\propto\pi L$; логарифмируем и раскрываем квадраты:

$$-2\log = \frac{(\theta-\theta_0)^2}{\tau^2}+t(\theta-y)^2 = \underbrace{\left(\frac1{\tau^2}+t\right)}_{=:A}\theta^2-2\underbrace{\left(\frac{\theta_0}{\tau^2}+ty\right)}_{=:B}\theta+\text{const}$$

Дополняем до полного квадрата: $A\theta^2-2B\theta=A(\theta-B/A)^2+\text{const}$. Значит апостериор нормален с точностью $A$ и средним $B/A$:

$$\frac1{\sigma_p^2}=\frac1{\tau^2}+t,\qquad \mu_p=\sigma_p^2\left(\frac{\theta_0}{\tau^2}+ty\right)$$

Подставим $ty=t\cdot(z_t/\sqrt t)=\sqrt t z_t$:

$$\boxed{\frac1{\sigma_p^2}=\frac1{\tau^2}+t,\qquad \mu_p=\sigma_p^2\left(\frac{\theta_0}{\tau^2}+\sqrt t\,z_t\right)}$$

* Точность $\pi(\theta)$ = точность prior + точность данных.
* $\mu_p$ — взвешенная сумма prior и данных с весами-точностями.
</details>

<details>
<summary>Замкнутость PPoS </summary>

Числитель CP линеен по $\theta$. Перепишем с $\theta=\mu_p+W$, где $W\sim\mathcal N(0,\sigma_p^2)$:

$$z_{1-\alpha}-\sqrt t \cdot z_t-\theta(1-t)=z_{1-\alpha}-\sqrt t \cdot z_t-\mu_p(1-t)-W(1-t)=:A-(1-t)W.$$

Обозначим $\widetilde W=(1-t)W\sim\mathcal N\big(0,(1-t)^2\sigma_p^2\big)$, тогда

$$\mathrm{CP}(\theta)=1-\Phi\left(\frac{A-\widetilde W}{\sqrt{1-t}}\right)$$

Для $W\sim\mathcal N(0,\sigma_W^2)$ и независимого $U\sim\mathcal N(0,1)$:

$$\Phi\left(\frac{a-W}{b}\right)=P\left(U\le\frac{a-W}{b}\right).$$

Беря матожидание по $W$ (теорема полной вероятности):

$$\mathbb E_W\left[\Phi\left(\frac{a-W}{b}\right)\right]=P\left(U\le\frac{a-W}{b}\right)=P(bU+W\le a)=\Phi\left(\frac{a}{\sqrt{b^2+\sigma_W^2}}\right),$$

так как $bU+W\sim\mathcal N(0,\,b^2+\sigma_W^2)$.

Здесь $a=A$, $b=\sqrt{1-t}$, $\sigma_W^2=(1-t)^2\sigma_p^2$:

$$\boxed{\mathrm{PPoS}=1-\Phi\left(\frac{z_{1-\alpha}-\sqrt t\,z_t-\mu_p(1-t)}{\sqrt{(1-t)+(1-t)^2\sigma_p^2}}\right)}$$
</details>

In [9]:
def plot_ppos(z_t=0.4, t=0.5, theta0=0.0, tau=0.8):
    """PPoS = E_π[CP(θ)], апостериор и CP(θ) на одном полотне."""
    # --- апостериор (замкнутая форма, нормальный prior) ---
    sigma2 = 1.0 / (1.0 / tau**2 + t)
    mu = sigma2 * (theta0 / tau**2 + np.sqrt(t) * z_t)

    theta_grid = np.linspace(-0.5, 4, 400)
    post = norm.pdf(theta_grid, loc=mu, scale=np.sqrt(sigma2))
    cp_theta = conditional_power(z_t, t, theta_grid, Z_ALPHA)
    pp = ppos(z_t, t, theta0, tau, Z_ALPHA)
    cp_trend = conditional_power(z_t, t, z_t / np.sqrt(t), Z_ALPHA)

    # --- нормируем шкалы, чтобы плотность и вероятность читались вместе ---
    post_scaled = post / max(post.max(), 1e-12) * 0.9
    cp_weighted = cp_theta * post
    cp_weighted_scaled = cp_weighted / max(cp_weighted.max(), 1e-12) * 0.9

    fig = go.Figure()

    # апостериор (нормированная плотность, заливка)
    fig.add_trace(go.Scatter(
        x=theta_grid, y=post_scaled, mode="lines",
        line=dict(color="rgba(120,120,120,0.9)", width=2.5),
        fill="tozeroy", fillcolor="rgba(153,153,153,0.15)",
        name="апостериор π(θ|данные) (норм.)",
    ))

    # CP(θ)
    fig.add_trace(go.Scatter(
        x=theta_grid, y=cp_theta, mode="lines",
        line=dict(color="rgba(31,119,180,1)", width=2.5),
        name="CP(θ)",
    ))

    # CP(θ)·π(θ) — нормированная заливка
    fig.add_trace(go.Scatter(
        x=theta_grid, y=cp_weighted_scaled, mode="lines",
        line=dict(color="rgba(217,95,2,0.9)", width=2),
        fill="tozeroy", fillcolor="rgba(217,95,2,0.25)",
        name="CP(θ)·π(θ) (норм.)",
    ))

    # вертикальные референсы
    fig.add_vline(
        x=mu, line_dash="dash", line_color="black",
        annotation_text=f"μ = {mu:.2f}",
        annotation_position="top",
    )
    fig.add_vline(
        x=design.theta_mde, line_dash="dot", line_color="rgba(31,119,180,1)",
        annotation_text=f"θ<sub>MDE</sub> = {design.theta_mde:.2f}",
        annotation_position="bottom",
    )

    # горизонтальные референсы
    fig.add_hline(
        y=pp, line_dash="dash", line_color="rgba(217,95,2,1)",
        annotation_text=f"PPoS = {pp:.3f}",
        annotation_position="right",
    )
    fig.add_hline(
        y=cp_trend, line_dash="dash", line_color="rgba(120,120,120,1)",
        annotation_text=f"CP(тренд) = {cp_trend:.3f}",
        annotation_position="right",
    )

    fig.update_layout(
        autosize=True,
        height=480,
        margin=dict(l=70, r=90, t=70, b=60),
        uirevision="ppos_plot",
        title=dict(
            text=(f"PPoS = E<sub>π</sub>[CP(θ)] = {pp:.3f}  "
                  f"(<i>z</i><sub>t</sub> = {z_t:.2f}, <i>t</i> = {t:.2f}, θ<sub>0</sub> = {theta0:.2f}, τ = {tau:.2f})"),
            x=0.5, xanchor="center",
        ),
        xaxis=dict(
            title="θ",
            dtick=0.5,
            tickformat=".1f",
            showgrid=True,
            zeroline=False,
            range=[theta_grid[0], theta_grid[-1]],
        ),
        yaxis=dict(
            title="плотность / вероятность",
            dtick=0.1,
            tickformat=".1f",
            showgrid=True,
            zeroline=False,
            range=[0, 1.0],
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom", y=1.02,
            xanchor="center", x=0.5,
            bgcolor="rgba(255,255,255,0.7)",
        ),
        hovermode="x unified",
    )

    return go.FigureWidget(fig)


interact(
    plot_ppos,
    z_t=FloatSlider(value=0.4, min=-3, max=3, step=0.05, description="zₜ"),
    t=FloatSlider(value=0.5, min=0.05, max=0.95, step=0.05, description="t"),
    theta0=FloatSlider(value=0.0, min=-1, max=3, step=0.1, description="prior θ₀"),
    tau=FloatSlider(value=0.8, min=0.05, max=3, step=0.05, description="prior τ"),
)

interactive(children=(FloatSlider(value=0.4, description='zₜ', max=3.0, min=-3.0, step=0.05), FloatSlider(valu…

<function __main__.plot_ppos(z_t=0.4, t=0.5, theta0=0.0, tau=0.8)>

### 1.4. $\beta$-spending

#### 1.4.1. Предварительная фиксация futility-границы

<details>
<summary>Логика симметрии с $\alpha$-spending</summary>

* $\alpha$-spending распределяет бюджет ошибки I рода по взглядам под $H_0$;
* $\beta$-spending — то же для ошибки II рода: «распределить $\beta$ так, чтобы суммарная вероятность зря остановить тест при реальном эффекте была не больше $\beta$».

«Зря остановить» возможно только когда эффект есть, поэтому futility-границу считают под $H_1$ ($\theta=\theta_{\mathrm{MDE}}$), а efficacy — под $H_0$.

Спендинг-функции — неубывающие функции доли информации $t$: $\alpha(0)=\beta(0)=0$, $\alpha(1)=\alpha$, $\beta(1)=\beta$.
В примерах ниже будем использовать степенное семейство $\beta(t)=\beta t^{\rho}$: $\rho=3$ — O'Brien–Fleming-подобная (консервативно рано), $\rho=1$ — Pocock-подобная (агрессивно рано).

</details>

<details>
<summary>Рекурсивный алгоритм расчёта</summary>

Введем следующие обозначения
* приращения бюджетов ошибок:

$$\Delta\alpha_k=\alpha(t_k)-\alpha(t_{k-1}),\qquad \Delta\beta_k=\beta(t_k)-\beta(t_{k-1}).$$

* $p_k(x)$ — (суб)плотность $B(t_k)$ среди траекторий, которые «дожили» до взгляда $k-1$, не пересекая ни одну границу. То есть $\int p_k(x)dx$ — вероятность ещё не остановиться к этому моменту.

<br>

Алгоритм состоит из 3 шагов и строит рекурсивно границу

**Шаг 1. Инициализация (считаем безусловную плотность $B(t_1)$)**
  
  $p_1(x)$ — это плотность положения процесса $B(t_1)$ в момент первого взгляда, при условии, что до этого момента мы никого не отбрасывали. А раз никого не отбрасывали — это просто безусловное распределение $B(t_1)$, и вся масса на месте: $\int p_1=1$.

- Efficacy-рекурсия стартует с $p_1^{eff}(x)=\varphi(x;0,t_1)$ — это $B(t_1)\sim\mathcal N(0,t_1)$ под $H_0$.
- Futility-рекурсия стартует с $p_1^{fut}(x)=\varphi(x;\theta t_1,t_1)$ — это $B(t_1)\sim\mathcal N(\theta t_1,t_1)$ под $H_1$.

Визуально две стартовые плотности выглядят так:

![Инициализация: две стартовые плотности $B(t_1)$](assets/l10_init.png)


**Шаг 2. Отсечения (взгляд $k$)**

* Обрезаем Efficacy $\int_{e_k\sqrt{t_k}}^{\infty}p_k^{eff}(x)\,dx=\Delta\alpha_k$
* Обрезаем Futility $\int_{-\infty}^{f_k\sqrt{t_k}}p_k^{fut}(x)\,dx=\Delta\beta_k$

<details>
<summary>Подробнее</summary>

* Efficacy — отрезаем правый хвост под $H_0$.

Ошибка I рода = «объявили эффект, когда его нет». Под $H_0$ дрейф равен нулю, поэтому процесс $B$ «топчется» около нуля; ложноположительный исход — это траектория, которая забралась аномально высоко вверх (выше $e_k\sqrt{t_k}$). Вероятность такого события на взгляде $k$ — это масса правого хвоста, и мы приравниваем её к приращению бюджета:

$$\int_{e_k\sqrt{t_k}}^{\infty}p_k^{eff}(x)\,dx=\Delta\alpha_k$$


* Futility — отрезаем левый хвост под $H_1$.

Ошибка II рода здесь = «зря остановили тест, когда эффект на самом деле есть». Под $H_1$ дрейф положителен ($\theta t_k$), поэтому «эффект есть» означает, что процесс в среднем дрейфует вверх. «Ошибочно выглядит слабо» — это траектория, которая оказалась аномально низко (ниже $f_k\sqrt{t_k}$). Вероятность такого события — масса левого хвоста:

$$\int_{-\infty}^{f_k\sqrt{t_k}}p_k^{fut}(x)\,dx=\Delta\beta_k$$

</details>

![Отсечение: правый хвост для efficacy, левый — для futility](assets/l10_trunc.png)



**Шаг 3. Свёртка**
  
Оставляем только траектории в области продолжения и добавляем независимое приращение $B(t_{k+1})-B(t_k)\sim\mathcal N(\theta\Delta_{k+1},\Delta_{k+1})$, $\Delta_{k+1}=t_{k+1}-t_k$. Futility-рекурсия идёт под $H_1$ (дрейф $\theta$), efficacy-рекурсия — под $H_0$ (дрейф $0$). В non-binding варианте (по умолчанию в коде) efficacy-рекурсия не учитывает futility-стопы, поэтому нижний предел у неё $-\infty$; в binding он равен $f_k\sqrt{t_k}$ (см. §1.4.2):

$$p_{k+1}^{fut}(y)=\int_{f_k\sqrt{t_k}}^{e_k\sqrt{t_k}}p_k^{fut}(x)\varphi\big(y;\,x+\theta\Delta_{k+1},\Delta_{k+1}\big)dx$$
$$p_{k+1}^{eff}(y)=\int_{-\infty}^{e_k\sqrt{t_k}}p_k^{eff}(x)\varphi\big(y;\,x,\Delta_{k+1}\big)dx$$

<details>
<summary>Подробнее</summary>

Т.е. для каждой возможной текущей позиции $x$ берем гауссиану с центром $x+\theta\Delta$ и дисперсией $\Delta$, взвешенную тем, сколько траекторий находится в точке $x$; затем складываем все эти гауссианы и получаем новую плотность $p_{k+1}$.

После отсечения на взгляде $k$ выжившие траектории распределены как $p_k(x)$, ограниченная областью продолжения $(f_k\sqrt{t_k},\,e_k\sqrt{t_k}]$. Каждая выжившая траектория, находящаяся в точке $x$, на следующем шаге делает случайный независимый шаг:

$$\text{следующая позиция} = x + \varepsilon,\qquad \varepsilon = B(t_{k+1})-B(t_k)\sim\mathcal N(\theta\Delta,\Delta).$$

То есть мы буквально прибавляем $\varepsilon$ к позиции $x$ каждой траектории. Шаг $\varepsilon$ независим от прошлого — это то самое свойство из §1.1 (блок «Переход к $B(t)$»), ради которого вводили $B$. Раз он независим, его распределение зависит только от длины интервала $\Delta$, а не от всей истории.

Отсюда и формула: траектория в точке $x$ с весом $p_k(x)dx$ порождает маленькую гауссиану $\varphi(y;x+\theta\Delta,\Delta)$, а новая плотность — интеграл по всем $x$:

$$p_{k+1}(y)=\int_{f_k\sqrt{t_k}}^{e_k\sqrt{t_k}}p_k(x)\,\varphi\big(y;\,x+\theta\Delta,\Delta\big)\,dx.$$

В результате форма распределения сдвигается вправо на $\theta\Delta$ (дрейф накопился) и размазывается на $\Delta$ (шум накопился). Причём раз урезали $p_k$ до области продолжения, масса, вытекшая за границы, потеряна — поэтому $p_{k+1}$ интегрируется уже в меньшую величину (новую вероятность выживания). Каждый шаг рекурсии буквально «вычёркивает» остановившихся и «протаскивает» выживших на один взгляд вперёд.

</details>

![Свёртка: каждая выжившая точка $x$ порождает гауссиану, их сумма и есть $p_{k+1}$](assets/l10_conv.png)


**Шаг 4. Возврат в $Z$-шкалу.** Рекурсия даёт границы в $B$-шкале — это $e_k\sqrt{t_k}$ и $f_k\sqrt{t_k}$. Делением на $\sqrt{t_k}$ получаем границы в $Z$-шкале — $e_k$ и $f_k$:
$$Z_k>e_k\iff B(t_k)>e_k\sqrt{t_k},\qquad Z_k<f_k\iff B(t_k)<f_k\sqrt{t_k}.$$

Почему это работает. Урезание оставляет только «ещё продолжающиеся» траектории, а свёртка прибавляет следующий независимый шаг. Независимость приращений гарантирует, что распределение следующего шага зависит только от текущей позиции, а не от всей истории, — поэтому рекурсия точна. Каждый шаг «тратит» ровно приращение бюджета $\Delta\alpha_k$ или $\Delta\beta_k$; суммируясь по $k$, они дают в точности $\alpha$ и $\beta$.

<details>
<summary>Анимация: вся рекурсия целиком</summary>

![Анимация рекурсии: отсечение и свёртка по взглядам](assets/l10_recursion.gif)

Оранжевое (слева от $f_k\sqrt{t_k}$) — futility, зелёное (справа от $e_k\sqrt{t_k}$) — efficacy, серое — продолжаем. Общая масса тает: $1{,}00\to0{,}97\to0{,}76\to0{,}43\to0{,}21$; доли futility-стопов по взглядам суммируются ровно в $\beta=0{,}20$, доли efficacy-стопов — в мощность $\approx0{,}78$.

</details>

</details>

In [10]:
def alpha_spending(t, alpha=0.05, family="obf"):
    t = np.asarray(t, dtype=float)
    if family == "obf":
        return 2 - 2 * norm.cdf(norm.ppf(1 - alpha / 2) / np.sqrt(t))
    if family == "pocock":
        return alpha * np.log(1 + (np.e - 1) * t)
    raise ValueError(family)


def beta_spending(t, beta=0.20, family="obf"):
    t = np.asarray(t, dtype=float)
    if family == "obf":
        return beta * t ** 3
    if family == "pocock":
        return beta * t
    raise ValueError(family)


def _convolve(p, grid, loc, scale):
    dx = grid[1] - grid[0]
    K = norm.pdf(grid[:, None], loc=grid[None, :] + loc, scale=scale)
    return (K @ p) * dx


def _quantile(p, grid, target, tail):
    dx = grid[1] - grid[0]
    cdf = np.cumsum(p) * dx
    vals = (cdf[-1] - cdf) if tail else cdf
    return grid[int(np.argmin(np.abs(vals - target)))]


def group_sequential_boundaries(t, alpha=0.05, beta=0.20, theta=None,
                                fam_alpha="obf", fam_beta="obf", grid_dx=0.01, n_sigma=6.0, binding=False):
    """Односторонние границы в Z-шкале: efficacy e_k (α-spending, под H0) и futility f_k (β-spending, под H1).

    Рекурсия идёт в B-шкале: eb_k = e_k·sqrt(t_k), fb_k = f_k·sqrt(t_k). Возвращает (eff, fut).
    """
    t = np.asarray(t, dtype=float)
    K = len(t)
    if theta is None:
        theta = norm.ppf(1 - alpha) + norm.ppf(1 - beta)
    dt = np.diff(np.concatenate(([0.0], t)))
    grid = np.arange(-n_sigma, n_sigma + grid_dx, grid_dx)
    d_alpha = np.diff(np.concatenate(([0.0], alpha_spending(t, alpha, fam_alpha))))
    d_beta = np.diff(np.concatenate(([0.0], beta_spending(t, beta, fam_beta))))
    eff = np.empty(K); fut = np.empty(K)
    p_eff = norm.pdf(grid, loc=0.0, scale=math.sqrt(dt[0]))
    p_fut = norm.pdf(grid, loc=theta * dt[0], scale=math.sqrt(dt[0]))
    for k in range(K):
        eb_k = _quantile(p_eff, grid, d_alpha[k], tail=True)
        eff[k] = eb_k / math.sqrt(t[k])
        fb_k = _quantile(p_fut, grid, d_beta[k], tail=False)
        fut[k] = fb_k / math.sqrt(t[k])
        if k < K - 1:
            d = dt[k + 1]
            # binding: траектории, остановленные futility, не тратят α (их убираем и из H0-плотности)
            keep_eff = ((grid > fb_k) & (grid <= eb_k)) if binding else (grid <= eb_k)
            p_eff = _convolve(p_eff * keep_eff, grid, 0.0, math.sqrt(d))
            p_fut = _convolve(p_fut * ((grid > fb_k) & (grid <= eb_k)), grid, theta * d, math.sqrt(d))
    return eff, fut


eff_bound, fut_bound = group_sequential_boundaries(design.t, design.alpha, design.beta, design.theta_mde)

In [11]:
tt = np.asarray(design.t)
bottom, top = -4.2, 4.2

fig = go.Figure()
# зона futility (ниже f_k) и зона efficacy (выше e_k)
fig.add_trace(go.Scatter(x=np.r_[tt, tt[-1], tt[0]], y=np.r_[fut_bound, bottom, bottom],
                         fill="toself", fillcolor="rgba(217,95,2,0.14)", line=dict(width=0),
                         hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scatter(x=np.r_[tt, tt[-1], tt[0]], y=np.r_[eff_bound, top, top],
                         fill="toself", fillcolor="rgba(27,158,119,0.14)", line=dict(width=0),
                         hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scatter(x=tt, y=eff_bound, mode="lines+markers", line=dict(color=C_EFF, width=2.8), name="efficacy <i>e</i><sub>k</sub>"))
fig.add_trace(go.Scatter(x=tt, y=fut_bound, mode="lines+markers", line=dict(color=C_FUT, width=2.8), name="futility <i>f</i><sub>k</sub>"))
fig.add_annotation(x=0.66, y=3.4, text="СТОП — эффект", showarrow=False, font=dict(color=C_EFF, size=13))
fig.add_annotation(x=0.66, y=-2.4, text="СТОП — futility", showarrow=False, font=dict(color=C_FUT, size=13))
fig.add_annotation(x=0.66, y=0.9, text="продолжаем", showarrow=False, font=dict(size=13))
fig = style(fig, height=480, title="Двухграничный групповой-последовательный дизайн",
            xlabel="доля информации <i>t</i>", ylabel="<i>Z</i>(<i>t</i>)")
fig.update_yaxes(range=[bottom, top])
fig


Включение futility так же легко распространяется на двусторонний тест:
* Для efficacy успехом считаем $\lvert Z_k\rvert>e_k$ (два симметричных хвоста)
* Для futility — $\lvert Z_k\rvert<f_k$ (узкий коридор вокруг нуля)





In [12]:
def two_sided_boundaries(t, alpha=0.05, beta=0.20, theta=None,
                         fam_alpha="obf", fam_beta="obf", dx=0.01, L=7.0):
    """Двусторонние границы в Z-шкале.

    efficacy: |Z_t| > e_k, суммарная ошибка I рода alpha (по alpha/2 на хвост), считается под H0;
    futility:  |Z_t| < f_k (центральный коридор), бюджет beta считается под H1 (theta).
    Рекурсия идёт в B-шкале (eb = e_k·sqrt(t_k), fb = f_k·sqrt(t_k)). Возвращает (eff, fut).
    """
    t = np.asarray(t, dtype=float)
    K = len(t)
    if theta is None:
        theta = norm.ppf(1 - alpha / 2) + norm.ppf(1 - beta)
    dt = np.diff(np.concatenate(([0.0], t)))
    grid = np.arange(-L, L + dx, dx)
    absg = np.abs(grid)
    d_alpha = np.diff(np.concatenate(([0.0], alpha_spending(t, alpha, fam_alpha))))
    d_beta = np.diff(np.concatenate(([0.0], beta_spending(t, beta, fam_beta))))

    def solve(mass, target, increasing):
        """Бисекция: находим x, при котором mass(x) = target."""
        lo, hi = 0.0, L
        for _ in range(60):
            mid = 0.5 * (lo + hi)
            if (mass(mid) < target) == increasing:
                lo = mid
            else:
                hi = mid
        return 0.5 * (lo + hi)

    def convolve(p, drift, d):
        kernel = norm.pdf(grid[:, None], loc=grid[None, :] + drift, scale=math.sqrt(d))
        return (kernel @ p) * dx

    p_eff = norm.pdf(grid, loc=0.0, scale=math.sqrt(dt[0]))             # H0
    p_fut = norm.pdf(grid, loc=theta * dt[0], scale=math.sqrt(dt[0]))   # H1
    eb = np.empty(K); fb = np.empty(K)
    for k in range(K):
        # efficacy: масса |x| > eb_k равна dAlpha_k
        eb[k] = solve(lambda x: p_eff[absg > x].sum() * dx, d_alpha[k], increasing=False)
        # futility: масса |x| < fb_k равна dBeta_k
        fb[k] = solve(lambda x: p_fut[absg < x].sum() * dx, d_beta[k], increasing=True)
        if k < K - 1:
            d = dt[k + 1]
            p_eff = convolve(p_eff * (absg <= eb[k]), 0.0, d)
            p_fut = convolve(p_fut * ((absg > fb[k]) & (absg <= eb[k])), theta * d, d)
    sqrt_t = np.sqrt(t)
    return eb / sqrt_t, fb / sqrt_t


eff2, fut2 = two_sided_boundaries(design.t, alpha=design.alpha, beta=design.beta)
tt = np.asarray(design.t)
bottom, top = -4.2, 4.2

fig = go.Figure()
# efficacy: два внешних хвоста (|Z| > e_k)
fig.add_trace(go.Scatter(x=np.r_[tt, tt[-1], tt[0]], y=np.r_[eff2, top, top],
                         fill="toself", fillcolor="rgba(27,158,119,0.18)", line=dict(width=0),
                         hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scatter(x=np.r_[tt, tt[-1], tt[0]], y=np.r_[-eff2, bottom, bottom],
                         fill="toself", fillcolor="rgba(27,158,119,0.18)", line=dict(width=0),
                         hoverinfo="skip", showlegend=False))
# futility: центральный коридор (|Z| < f_k), то есть конус около нуля
fig.add_trace(go.Scatter(x=np.r_[tt, tt[::-1]], y=np.r_[fut2, -fut2[::-1]],
                         fill="toself", fillcolor="rgba(217,95,2,0.20)", line=dict(width=0),
                         hoverinfo="skip", showlegend=False))
# границы
fig.add_trace(go.Scatter(x=tt, y=eff2, mode="lines+markers", line=dict(color=C_EFF, width=2.8), name="efficacy ±<i>e</i><sub>k</sub>"))
fig.add_trace(go.Scatter(x=tt, y=-eff2, mode="lines+markers", line=dict(color=C_EFF, width=2.8), showlegend=False))
fig.add_trace(go.Scatter(x=tt, y=fut2, mode="lines+markers", line=dict(color=C_FUT, width=2.4, dash="dot"), name="futility ±<i>f</i><sub>k</sub>"))
fig.add_trace(go.Scatter(x=tt, y=-fut2, mode="lines+markers", line=dict(color=C_FUT, width=2.4, dash="dot"), showlegend=False))
# подписи зон
fig.add_annotation(x=0.62, y=3.5, text="ОТВЕРГАЕМ H<sub>0</sub><br>эффект > 0  (<i>Z</i> > +<i>e</i><sub>k</sub>)", showarrow=False, font=dict(color=C_EFF, size=12))
fig.add_annotation(x=0.62, y=-3.5, text="ОТВЕРГАЕМ H<sub>0</sub><br>эффект < 0  (<i>Z</i> < −<i>e</i><sub>k</sub>)", showarrow=False, font=dict(color=C_EFF, size=12))
fig.add_annotation(x=0.85, y=0.0, text="FUTILITY<br>|<i>Z</i>| < <i>f</i><sub>k</sub><br>(эффект ≈ 0)", showarrow=False, font=dict(color=C_FUT, size=12))
fig.add_annotation(x=0.30, y=1.6, text="продолжаем", showarrow=False, font=dict(color="#555555", size=11))
fig.add_annotation(x=0.30, y=-1.6, text="продолжаем", showarrow=False, font=dict(color="#555555", size=11))
fig = style(fig, height=520, title="Двусторонний дизайн: пять зон на каждом взгляде",
            xlabel="доля информации <i>t</i>", ylabel="<i>Z</i>(<i>t</i>)")
fig.update_yaxes(range=[bottom, top])
fig


#### 1.4.2. Binding и non-binding фиксация

Futility-граница влияет на то, какие траектории доживают до конца. Вопрос: как учитывать её при расчёте $\alpha$?

- **Binding**: futility-стоп обязателен, $\alpha$ считается с удалением траекторий ниже $f_k$. Если проигнорировать стоп — $\alpha$ вырастет.
- **Non-binding**: $\alpha$ считается так, *будто futility-границы нет*. Стоп можно отменить без риска для $\alpha$.

В случае binding границ итоговая $\alpha$ не превышает получаемую ошибку первого рода при non-binding: в последнем варианте область продолжения — «ещё не пересекли efficacy» ($-\infty<Z\le e_j$); в binding она дополнительно ограничена снизу ($f_j<Z\le e_j$). Продолжающееся множество в binding — подмножество, поэтому масса, достигающая efficacy, не больше.

График ниже показывает, как это меняет границы (для наглядности взят агрессивный $\beta$-spending Pocock-типа). Futility-граница $f_k$ получается одной и той же: она считается под $H_1$ и от выбора подхода почти не зависит. Меняется efficacy-граница $e_k$: при non-binding бюджет $\alpha$ тратится на все траектории, при binding — только на выжившие после futility-стопа, поэтому тот же бюджет даёт более низкую $e_k$ (заливка — выигрыш binding). Цена — обязательность стопа: если взять binding-границу $e_k$, а futility-стоп потом отменить, $\alpha$ превысит номинал (см. вывод ячейки).


In [13]:
def overall_alpha_upper(eff, t, fut=None, n_sim=400000):
    """Монте-Карло: доля H0-траекторий, пересёкших efficacy e_k (с futility-стопом по f_k, если fut задан)."""
    dt = np.diff(np.concatenate(([0.0], t)))
    B = np.cumsum(rng.normal(0, np.sqrt(dt), size=(n_sim, len(t))), axis=1)
    Z = B / np.sqrt(t)
    active = np.ones(n_sim, dtype=bool)
    crossed = np.zeros(n_sim, dtype=bool)
    for k in range(len(t)):
        if fut is not None:
            active &= ~(Z[:, k] < fut[k]) & ~crossed
        crossed |= active & (Z[:, k] > eff[k])
        active &= ~crossed
    return crossed.mean()


# Для наглядности берём агрессивный β-spending Pocock-типа: чем больше траекторий режет futility,
# тем заметнее разница между подходами.
fam_b = "pocock"
t = np.asarray(design.t)
eff_nb, fut_nb = group_sequential_boundaries(design.t, design.alpha, design.beta, design.theta_mde,
                                             fam_beta=fam_b, binding=False)
eff_bd, fut_bd = group_sequential_boundaries(design.t, design.alpha, design.beta, design.theta_mde,
                                             fam_beta=fam_b, binding=True)

# Проверка: α каждой efficacy-границы при обязательном futility-стопе и без него
alpha_nb_ignore = overall_alpha_upper(eff_nb, t, fut=None)    # non-binding границы, futility игнорируется
alpha_nb_stop = overall_alpha_upper(eff_nb, t, fut=fut_nb)    # non-binding границы, futility обязателен
alpha_bd_stop = overall_alpha_upper(eff_bd, t, fut=fut_bd)    # binding границы, futility обязателен
alpha_bd_ignore = overall_alpha_upper(eff_bd, t, fut=None)    # binding границы, futility проигнорировали
print(f"non-binding e_k: α = {alpha_nb_ignore:.4f} (futility игнорируется), {alpha_nb_stop:.4f} (futility обязателен)")
print(f"binding e_k:     α = {alpha_bd_stop:.4f} (futility обязателен),    {alpha_bd_ignore:.4f} (futility проигнорировали)")
print(f"max |f_k(binding) − f_k(non-binding)| = {np.max(np.abs(fut_bd - fut_nb)):.3f}")

fig = make_subplots(rows=1, cols=2, column_widths=[0.55, 0.45], horizontal_spacing=0.09,
                    subplot_titles=("Все границы", "Увеличенный фрагмент: efficacy"))


def add_boundaries(col, show_legend):
    # область, на которую binding расширяет efficacy-зону
    fig.add_trace(go.Scatter(x=np.r_[t, t[::-1]], y=np.r_[eff_nb, eff_bd[::-1]], fill="toself",
                             fillcolor="rgba(27,158,119,0.30)", line=dict(width=0), hoverinfo="skip",
                             name="выигрыш binding", showlegend=show_legend), row=1, col=col)
    fig.add_trace(go.Scatter(x=t, y=eff_nb, mode="lines+markers", line=dict(color=C_EFF, width=2.8),
                             name="efficacy <i>e</i><sub>k</sub>, non-binding", showlegend=show_legend), row=1, col=col)
    fig.add_trace(go.Scatter(x=t, y=eff_bd, mode="lines+markers",
                             line=dict(color="#0b4f3c", width=2.8, dash="dash"),
                             name="efficacy <i>e</i><sub>k</sub>, binding", showlegend=show_legend), row=1, col=col)
    fig.add_trace(go.Scatter(x=t, y=fut_nb, mode="lines+markers", line=dict(color=C_FUT, width=2.8),
                             name="futility <i>f</i><sub>k</sub> (одинакова при обоих подходах)", showlegend=show_legend),
                  row=1, col=col)


add_boundaries(1, True)
add_boundaries(2, False)
fig.update_layout(height=500, template="plotly_white",
                  title=f"Границы при non-binding и binding фиксации (на последнем взгляде <i>e</i><sub>K</sub> = {eff_nb[-1]:.2f} против {eff_bd[-1]:.2f})",
                  legend=dict(orientation="h", yanchor="top", y=-0.18, xanchor="left", x=0),
                  margin=dict(l=60, r=20, t=90, b=90))
fig.update_xaxes(title_text="доля информации <i>t</i>", showgrid=True, gridcolor="rgba(0,0,0,0.07)")
fig.update_yaxes(title_text="<i>Z</i>(<i>t</i>)", col=1, showgrid=True, gridcolor="rgba(0,0,0,0.07)")
fig.update_yaxes(range=[-1.6, 3.6], col=1)
fig.update_xaxes(range=[0.45, 1.02], col=2)
fig.update_yaxes(range=[1.6, 2.7], col=2, showgrid=True, gridcolor="rgba(0,0,0,0.07)")
fig


non-binding e_k: α = 0.0501 (futility игнорируется), 0.0450 (futility обязателен)
binding e_k:     α = 0.0507 (futility обязателен),    0.0566 (futility проигнорировали)
max |f_k(binding) − f_k(non-binding)| = 0.010


## 2. Влияние futility на ошибки I и II рода

<details>
<summary>Почему futility НЕ раздувает $\alpha$, но снижает мощность (развернуть)</summary>

**Futility не раздувает $\alpha$.** Ошибка I рода — траектория, которая под $H_0$ забралась выше efficacy. Futility отбрасывает траектории *ниже* себя; они и так редко достигали бы efficacy, а стоп лишь уменьшает их число. Вероятность отвергнуть при $H_0$ может только уменьшиться.

**Futility снижает мощность.** Под $H_1$ часть траекторий на раннем взгляде случайно выглядит слабо — futility зря их останавливает. Это плата за экономию трафика.

</details>

Симулируем группово-последовательный дизайн и измеряем операционные характеристики. Ниже — результаты с человеческими названиями.


In [26]:
def simulate_trial(theta, t, eff, fut, n_sim=50000, futility="boundary", gamma=0.15,
                    theta_cp=None, theta0=0.0, tau=0.8, z_alpha=1.6449):
    """Симуляция группового-последовательного дизайна.

    eff, fut: границы e_k и f_k в Z-шкале.
    futility: "none" | "boundary" (фиксированная f_k) | "cp" (CP<gamma) | "ppos" (PPoS<gamma).
    Исходы: efficacy / futility / complete; возвращает исход и долю информации на остановке.
    """
    dt = np.diff(np.concatenate(([0.0], t)))
    B = np.cumsum(rng.normal(theta * dt, np.sqrt(dt), size=(n_sim, len(t))), axis=1)
    Z = B / np.sqrt(t)
    outcome = np.full(n_sim, "complete")
    stop_frac = np.ones(n_sim)
    for k in range(len(t)):
        active = outcome == "complete"
        if not active.any():
            break
        stop_eff = active & (Z[:, k] > eff[k])
        still = active & ~stop_eff
        if k == len(t) - 1:
            # финальный взгляд: futility = Z < f_K (фиксированная граница);
            # CP/PPoS тут вырождены (t=1 → sqrt(1-t)=0), поэтому их не считаем.
            stop_fut = still & (Z[:, k] < fut[k])
        elif futility == "boundary":
            stop_fut = still & (Z[:, k] < fut[k])
        elif futility == "cp":
            cp = conditional_power(Z[still, k], t[k], theta_cp, z_alpha)
            stop_fut = still.copy(); stop_fut[still] = cp < gamma
        elif futility == "ppos":
            pp = ppos(Z[still, k], t[k], theta0, tau, z_alpha)
            stop_fut = still.copy(); stop_fut[still] = pp < gamma
        else:
            stop_fut = np.zeros_like(still)
        outcome[stop_eff] = "efficacy"; outcome[stop_fut] = "futility"
        stop_frac[stop_eff] = t[k]; stop_frac[stop_fut] = t[k]
    return outcome, stop_frac


def oc(theta, t, eff, fut, n_sim=50000, **kw):
    out, stop = simulate_trial(theta, t, eff, fut, n_sim, **kw)
    return {
        "FPR / power": (out == "efficacy").mean(),
        "P(futility stop)": (out == "futility").mean(),
        "P(дошли до конца)": (out == "complete").mean(),
        "E[N]/N_max": stop.mean(),
    }


In [27]:
t = np.asarray(design.t)
n_sim = 50000

rows = []
scenarios = [
    ("H₀: эффекта нет (θ = 0)", 0.0, "ошибка I рода (α)"),
    ("H₁: эффект = MDE (θ = θ_MDE)", design.theta_mde, "мощность (1 − β)"),
]
for label, theta, meaning in scenarios:
    r = oc(theta, t, eff_bound, fut_bound, n_sim)
    rows.append({
        "Сценарий": label,
        "Остановка по efficacy (H₀ отвергнута)": r["FPR / power"],
        "Что это значит в сценарии": meaning,
        "Остановка по futility": r["P(futility stop)"],
        "Дошли до конца без остановки": r["P(дошли до конца)"],
        "Средняя доля данных до остановки": r["E[N]/N_max"],
    })

mc = pd.DataFrame(rows)
# доли показываем в процентах (строки, чтобы не зависеть от jinja2 / Styler)
share_cols = ["Остановка по efficacy (H₀ отвергнута)", "Остановка по futility",
              "Дошли до конца без остановки", "Средняя доля данных до остановки"]
for col in share_cols:
    mc[col] = mc[col].map("{:.1%}".format)
mc


,Сценарий,Остановка по efficacy (H₀ отвергнута),Что это значит в сценарии,Остановка по futility,Дошли до конца без остановки,Средняя доля данных до остановки
0,H₀: эффекта нет (θ = 0),5.0%,ошибка I рода (α),94.0%,1.0%,59.1%
1,H₁: эффект = MDE (θ = θ_MDE),77.1%,мощность (1 − β),19.7%,3.1%,68.9%


In [28]:
# Операционные характеристики по спектру эффектов: три доли, в сумме = 1 (stacked area).
thetas = np.linspace(-0.2, 3.0, 17)
N = 30000
out_df = pd.DataFrame([oc(th, t, eff_bound, fut_bound, N) for th in thetas])
out_df["θ"] = thetas

fig = go.Figure()
fig.add_trace(go.Scatter(x=thetas, y=out_df["FPR / power"], mode="lines", stackgroup="one",
                         line=dict(color=C_EFF, width=0), fillcolor="rgba(27,158,119,0.6)", name="efficacy (power/FPR)"))
fig.add_trace(go.Scatter(x=thetas, y=out_df["P(futility stop)"], mode="lines", stackgroup="one",
                         line=dict(color=C_FUT, width=0), fillcolor="rgba(217,95,2,0.6)", name="futility stop"))
fig.add_trace(go.Scatter(x=thetas, y=out_df["P(дошли до конца)"], mode="lines", stackgroup="one",
                         line=dict(color=C_CONT, width=0), fillcolor="rgba(153,153,153,0.5)", name="дошли до конца"))
fig.add_vline(x=design.theta_mde, line_dash="dash", line_color="black", annotation_text="θ<sub>MDE</sub>")
fig = style(fig, height=460,
            title="Операционные характеристики: три полосы в сумме дают 1 (при любом θ исход ровно один)",
            xlabel="истинный дрейф θ", ylabel="доля исходов")
fig


## 3. Сравнение всех методов


#### 3.1. Сквозной пример 


Теперь соберём всё вместе на конкретном примере. Метрика — **средний чек** ($\sigma$ варьируемо, контроль $\mu_0$ не влияет на стандартизованный результат). Сравним, как разные правила остановки ведут себя по мощности, ошибке I рода и экономии трафика.

Два замечания к таблице: **(1)** binding и non-binding здесь используют одни и те же границы — они различаются только расчётом $\alpha$ (у binding он ниже), мощность и ASN у них совпадают; **(2)** PPoS-futility со **скептическим** prior ($\theta_0=0$, $\tau=0.8$) очень консервативен — видно, как сильно prior влияет на результат; с «дизайновым» prior ($\theta_0=\theta_{\mathrm{MDE}}$) он близок к CP.

In [29]:
def compare_methods(sigma=100.0, mde=10.0, alpha=0.05, beta=0.20, K=5,
                     fam_alpha="obf", fam_beta="obf", gamma=0.15, theta0=0.0, tau=0.8, n_sim=20000):
    delta = mde / sigma
    theta_mde = norm.ppf(1 - alpha) + norm.ppf(1 - beta)
    z_alpha = norm.ppf(1 - alpha)
    t = np.linspace(0.3, 1.0, K)
    eff, fut = group_sequential_boundaries(t, alpha, beta, theta_mde, fam_alpha, fam_beta)

    methods = {}
    # 1) фиксированный z-тест (один взгляд в конце)
    methods["Фиксированный z-тест"] = dict(t=np.array([1.0]), eff=np.array([z_alpha]), fut=np.array([-np.inf]), futility="none")
    # 2) GST только efficacy
    methods["GST (efficacy only)"] = dict(t=t, eff=eff, fut=np.full(K, -np.inf), futility="none")
    # 3) GST + futility non-binding
    methods["GST + futility (non-binding)"] = dict(t=t, eff=eff, fut=fut, futility="boundary")
    # 4) GST + futility binding (те же границы, но α считаем binding)
    methods["GST + futility (binding)"] = dict(t=t, eff=eff, fut=fut, futility="boundary", binding=True)
    # 5) GST + CP-futility
    methods["GST + CP-futility"] = dict(t=t, eff=eff, fut=fut, futility="cp", gamma=gamma, theta_cp=theta_mde)
    # 6) GST + PPoS-futility
    methods["GST + PPoS-futility"] = dict(t=t, eff=eff, fut=fut, futility="ppos", gamma=gamma, theta0=theta0, tau=tau)

    rows = []
    for name, m in methods.items():
        r0 = oc(0.0, m["t"], m["eff"], m["fut"], n_sim, futility=m["futility"],
                gamma=m.get("gamma", 0.15), theta_cp=m.get("theta_cp", theta_mde),
                theta0=m.get("theta0", 0.0), tau=m.get("tau", 0.8))
        r1 = oc(theta_mde, m["t"], m["eff"], m["fut"], n_sim, futility=m["futility"],
                gamma=m.get("gamma", 0.15), theta_cp=m.get("theta_cp", theta_mde),
                theta0=m.get("theta0", 0.0), tau=m.get("tau", 0.8))
        fpr = r0["FPR / power"]
        if m.get("binding"):
            fpr = overall_alpha_upper(m["eff"], m["t"], fut=m["fut"], n_sim=200000)
        rows.append({
            "Метод": name,
            "FPR (H0)": fpr,
            "Мощность (H1)": r1["FPR / power"],
            "P(futility | H1)": r1["P(futility stop)"],
            "E[N]/N_max (H1)": r1["E[N]/N_max"],
        })
    df = pd.DataFrame(rows)
    return df, theta_mde


cmp_df, theta_mde = compare_methods()

# Читаемая версия таблицы для показа (cmp_df с исходными названиями используется в графиках ниже)
cmp_view = cmp_df.rename(columns={
    "FPR (H0)": "Ложные срабатывания при H₀ (ошибка I рода)",
    "Мощность (H1)": "Мощность при H₁ (эффект = MDE)",
    "P(futility | H1)": "Зря остановили по futility при H₁",
    "E[N]/N_max (H1)": "Средняя доля данных до остановки при H₁",
})
for col in cmp_view.columns[1:]:
    cmp_view[col] = cmp_view[col].map("{:.1%}".format)
cmp_view


,Метод,Ложные срабатывания при H₀ (ошибка I рода),Мощность при H₁ (эффект = MDE),Зря остановили по futility при H₁,Средняя доля данных до остановки при H₁
0,Фиксированный z-тест,5.1%,80.1%,0.0%,100.0%
1,GST (efficacy only),4.8%,79.0%,0.0%,77.5%
2,GST + futility (non-binding),5.0%,78.0%,19.6%,74.2%
3,GST + futility (binding),4.9%,77.2%,20.4%,74.1%
4,GST + CP-futility,4.9%,78.3%,19.2%,75.0%
5,GST + PPoS-futility,3.0%,57.9%,41.3%,55.8%


In [30]:
def plot_methods(df, title, height=520):
    """Сравнение методов: слева мощность и расход данных (оси 0–1), справа ошибка I рода (FPR)."""
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Bar(x=df["Метод"], y=df["Мощность (H1)"], name="Мощность",
                         marker_color=C_EFF, offsetgroup="power"), secondary_y=False)
    fig.add_trace(go.Bar(x=df["Метод"], y=df["E[N]/N_max (H1)"], name="Использовано данных (меньше = экономия)",
                         marker_color=C_CONT, offsetgroup="data"), secondary_y=False)
    # ошибка I рода — ромбики на правой оси (чтобы не путать масштаб с барами слева)
    fig.add_trace(go.Scatter(x=df["Метод"], y=[0.05] * len(df), mode="lines",
                             line=dict(color=C_FUT, width=1.5, dash="dot"),
                             name="номинальная α = 0.05 (правая ось)", hoverinfo="skip"), secondary_y=True)
    fig.add_trace(go.Scatter(x=df["Метод"], y=df["FPR (H0)"], mode="markers",
                             marker=dict(color=C_FUT, size=15, symbol="diamond", line=dict(color="white", width=1.5)),
                             name="Ошибка I рода, FPR (правая ось)"), secondary_y=True)
    fig = style(fig, height=height, title=title, xlabel="метод")
    fig.update_layout(barmode="group", bargap=0.25,
                      # легенда справа от графика, чтобы не налезала на заголовок
                      legend=dict(orientation="v", x=1.09, xanchor="left", y=1, yanchor="top"),
                      margin=dict(l=60, r=20, t=70, b=50))
    fig.update_xaxes(tickangle=-20)

    # левая ось: шаг 0.05, 20 делений
    fig.update_yaxes(title_text="Мощность / доля использованных данных", range=[0, 1], dtick=0.05,
                     tickformat=".2f", secondary_y=False)
    # правая ось (FPR): тоже 20 делений
    fpr_top = max(0.10, np.ceil(df["FPR (H0)"].max() * 20) / 20)
    fig.update_yaxes(title_text="Ошибка I рода (FPR)", range=[0, fpr_top], dtick=fpr_top / 20,
                     tickformat=".3f", showgrid=False, secondary_y=True)
    return fig


plot_methods(cmp_df, "Сравнение методов: мощность и расход данных (слева), ошибка I рода (справа)")


In [31]:
def plot_comparison(K=5, sigma=100.0, mde=10.0, fam_alpha="obf", fam_beta="obf", gamma=0.15):
    df, _ = compare_methods(sigma=sigma, mde=mde, K=K, fam_alpha=fam_alpha, fam_beta=fam_beta, gamma=gamma, n_sim=8000)
    fig = plot_methods(df, f"Сравнение методов (K={K}, σ={sigma}, MDE={mde})", height=500)
    return go.FigureWidget(fig)

interact(plot_comparison,
         K=IntSlider(value=5, min=1, max=50, step=1, description="взглядов K"),
         sigma=FloatSlider(value=100.0, min=10, max=500, step=10, description="σ метрики"),
         mde=FloatSlider(value=10.0, min=1, max=50, step=1, description="MDE"),
         fam_alpha=Dropdown(options=["obf", "pocock"], value="obf", description="α-spending"),
         fam_beta=Dropdown(options=["obf", "pocock"], value="obf", description="β-spending"),
         gamma=FloatSlider(value=0.15, min=0.05, max=0.40, step=0.05, description="порог CP"))


interactive(children=(IntSlider(value=5, description='взглядов K', max=50, min=1), FloatSlider(value=100.0, de…

<function __main__.plot_comparison(K=5, sigma=100.0, mde=10.0, fam_alpha='obf', fam_beta='obf', gamma=0.15)>

Сквозной пример выше сравнивает методы в двух точках — $H_0$ и $H_1$. Полезно посмотреть на **всю зависимость от истинного эффекта $\theta$**: futility-остановка — это «покупка экономии трафика за небольшую потерю мощности», и цена/выгода этой сделки сильно меняются с $\theta$.

Сравниваем два дизайна с **одинаковыми efficacy-границами** $e_k$ и одинаковым $N_{\max}$:

- **GST (efficacy only)** — останавливаемся только по efficacy;
- **GST + futility (non-binding)** — дополнительно останавливаемся по $|Z_k|<f_k$.

**Левая панель** — экономия каждого дизайна относительно базового (фиксированного) z-теста, который всегда использует весь $N_{\max}$: $1-\mathrm{ASN}/\mathrm{ASN}_{\text{base}}$, где $\mathrm{ASN}_{\text{base}}=1$. **Правая панель** — дополнительная экономия от futility относительно GST без futility ($\mathrm{ASN}_{\text{eff}}-\mathrm{ASN}_{\text{fut}}$) и потеря мощности в п.п. Вертикальные линии — $\theta=0$ ($H_0$) и $\theta=\theta_{\mathrm{MDE}}$ ($H_1$).


In [32]:
# Сравнение GST (efficacy only) и GST + non-binding futility как функция истинного θ.
# Левая панель: экономия относительно базового фиксированного z-теста (он всегда тратит весь N_max).
# Правая панель: дополнительная экономия от futility относительно GST без futility + потеря мощности.
t = np.asarray(design.t)
K = len(t)
eff_cmp, fut_cmp = eff_bound, fut_bound          # те же границы, что в разделе 2
fut_none = np.full(K, -np.inf)

thetas_cmp = np.round(np.arange(0.0, 2.0 * design.theta_mde + 1e-9, 0.05), 2)
n_sim_cmp = 20000

rows = []
for th in thetas_cmp:
    r_e = oc(th, t, eff_cmp, fut_none, n_sim_cmp, futility="none")
    r_f = oc(th, t, eff_cmp, fut_cmp, n_sim_cmp, futility="boundary")
    asn_e = r_e["E[N]/N_max"]
    asn_f = r_f["E[N]/N_max"]
    rows.append({
        "theta": th,
        "asn_eff": asn_e,
        "asn_fut": asn_f,
        # экономия относительно базового z-теста (его ASN = 1, он всегда тратит N_max)
        "savings_base_eff": 100 * (1 - asn_e),
        "savings_base_fut": 100 * (1 - asn_f),
        # дополнительная экономия futility относительно GST без futility
        "savings_pct": 100 * (asn_e - asn_f),
        "power_eff": r_e["FPR / power"],
        "power_fut": r_f["FPR / power"],
        "power_loss_pp": 100 * (r_e["FPR / power"] - r_f["FPR / power"]),
        "p_fut": r_f["P(futility stop)"],
    })
sweep = pd.DataFrame(rows)

fig = make_subplots(
    rows=1, cols=2, horizontal_spacing=0.11,
    specs=[[{}, {"secondary_y": True}]],
    subplot_titles=(
        "Экономия относительно базового z-теста",
        "Дополнительная экономия от futility и потеря мощности",
    ),
)

# --- левая панель: экономия каждого дизайна относительно базового z-теста ---
fig.add_trace(go.Scatter(x=sweep["theta"], y=sweep["savings_base_eff"], mode="lines",
                         line=dict(color=C_CONT, width=2.6), name="GST (efficacy only)"),
              row=1, col=1)
fig.add_trace(go.Scatter(x=sweep["theta"], y=sweep["savings_base_fut"], mode="lines",
                         line=dict(color=C_FUT, width=2.6), name="GST + futility"),
              row=1, col=1)

# --- правая панель: экономия futility относительно efficacy-only + потеря мощности ---
fig.add_trace(go.Scatter(x=sweep["theta"], y=sweep["savings_pct"], mode="lines",
                         fill="tozeroy", fillcolor="rgba(217,95,2,0.18)",
                         line=dict(color=C_FUT, width=2.6), name="экономия от futility, % от N_max"),
              row=1, col=2, secondary_y=False)
fig.add_trace(go.Scatter(x=sweep["theta"], y=sweep["power_loss_pp"], mode="lines",
                         line=dict(color=C_EFF, width=2.6, dash="dot"),
                         name="потеря мощности, п.п. (правая ось)"),
              row=1, col=2, secondary_y=True)

# вертикальные линии H0 / H1
for col in (1, 2):
    fig.add_vline(x=0.0, line_dash="dot", line_color="black", row=1, col=col,
                  annotation_text="H₀", annotation_position="top right")
    fig.add_vline(x=design.theta_mde, line_dash="dash", line_color="black", row=1, col=col,
                  annotation_text="H₁", annotation_position="top left")

fig.update_yaxes(title_text="экономия vs z-тест, %", range=[-2, 60], nticks=20, row=1, col=1)
fig.update_yaxes(title_text="экономия от futility, %", range=[-2, 42], nticks=20,
                 row=1, col=2, secondary_y=False)
fig.update_yaxes(title_text="потеря мощности, п.п.", range=[-0.2, 2.4], nticks=20,
                 row=1, col=2, secondary_y=True, showgrid=False)
fig.update_xaxes(title_text="истинный дрейф θ", nticks=20, row=1, col=1)
fig.update_xaxes(title_text="истинный дрейф θ", nticks=20, row=1, col=2)
fig.update_layout(height=520, template="plotly_white",
                  legend=dict(orientation="h", yanchor="top", y=-0.22),
                  margin=dict(l=60, r=70, t=70, b=110))
fig


### 3.2. Насколько страшно ошибиться с $\theta$: карта экономии ASN

График выше фиксировал дизайн под $\theta_{\mathrm{MDE}}$ и менял только **реальный** $\theta$. Здесь меняем **оба**:

- ось X — **ожидаемый** $\theta$, под которое строятся futility-границы (дизайн);
- ось Y — **реальный** $\theta$ (истинный эффект);
- цвет — экономия ASN от futility, % от $N_{\max}$.

Белая пунктирная диагональ — «правильный» дизайн ($\theta_{\text{ожид}}=\theta_{\text{реал}}$): вдоль неё futility стабильно экономит ~3–4%. Всё, что ниже диагонали, — **переоценили** $\theta$ (границы futility агрессивнее, экономия больше, но часть истинных эффектов остановится как futility — мощность падает). Всё, что выше диагонали, — **недооценили** $\theta$ (экономия меньше, но и мощность почти не страдает).

In [33]:
# Heatmap: экономия ASN от futility при разных (ожидаемый θ, реальный θ).
# Efficacy-границы от θ не зависят (считаются под H0), futility-границы строятся под каждым θ_design.
t = np.asarray(design.t)
K = len(t)
fut_none = np.full(K, -np.inf)

theta_design = np.round(np.linspace(1.0, 4.0, 71), 2)
theta_real = np.round(np.linspace(0.0, 4.5, 70), 2)
n_sim_h = 15000

# ASN efficacy-only зависит только от реального θ — считаем один раз
asn_eff_by_real = {r: oc(r, t, eff_bound, fut_none, n_sim_h, futility="none")["E[N]/N_max"]
                   for r in theta_real}

saving = np.zeros((len(theta_real), len(theta_design)))
for j, a in tqdm(enumerate(theta_design)):
    _, fut_a = group_sequential_boundaries(t, design.alpha, design.beta, a)
    for i, r in enumerate(theta_real):
        asn_f = oc(r, t, eff_bound, fut_a, n_sim_h, futility="boundary")["E[N]/N_max"]
        saving[i, j] = 100 * (asn_eff_by_real[r] - asn_f)

fig = go.Figure(go.Heatmap(
    z=saving, x=theta_design, y=theta_real,
    colorscale="Viridis",
    colorbar=dict(title="экономия, % от N_max"),
    hovertemplate="ожидаемый θ=%{x:.2f}<br>реальный θ=%{y:.2f}<br>экономия=%{z:.1f}%<extra></extra>",
))
diag = np.linspace(1.0, 4.0, 50)
fig.add_trace(go.Scatter(x=diag, y=diag, mode="lines",
                         line=dict(color="white", width=2, dash="dash"),
                         name="θ_ожид. = θ_реал. (правильный дизайн)"))
fig.update_xaxes(title_text="ожидаемый θ (дизайн futility)")
fig.update_yaxes(title_text="реальный θ")
fig.update_layout(height=560, template="plotly_white",
                  title="Экономия ASN от futility: ожидаемый θ vs реальный θ",
                  legend=dict(orientation="h", y=-0.12))
fig


71it [03:21,  2.84s/it]


### 3.3. Цена ошибки: падение мощности

Та же сетка (ожидаемый θ × реальный θ), но цвет — **падение мощности** от futility, п.п. Считаем парно на общих траекториях, поэтому значения $\ge 0$ (futility только убирает efficacy-остановки). Диагональ — «правильный» дизайн: около $\theta_{\text{MDE}}$ потеря ~1 п.п. Ниже диагонали (переоценили θ) потеря мощности резко растёт — до ~25 п.п.; выше диагонали (недооценили) мощность почти не страдает. Вместе с картой экономии это показывает сделку: переоценка θ «даёт» экономию ASN, но платит за неё мощностью.

In [34]:
# Heatmap: падение мощности от futility при разных (ожидаемый θ, реальный θ).
# Парное сравнение на общих B-траекториях: futility только убирает efficacy-остановки, поэтому loss >= 0.
t = np.asarray(design.t)
K = len(t)
fut_none = np.full(K, -np.inf)
theta_design = np.round(np.linspace(1.0, 4.0, 71), 2)
theta_real = np.round(np.linspace(0.0, 4.5, 70), 2)
n_sim_h = 15000
dt = np.diff(np.concatenate(([0.0], t)))
rng_h = np.random.default_rng(2024)

def power_from_B(eff, fut, B):
    Z = B / np.sqrt(t)
    out = np.full(B.shape[0], 2)
    for k in range(K):
        act = out == 2
        if not act.any():
            break
        se = act & (Z[:, k] > eff[k])
        st = act & ~se
        sf = st & (Z[:, k] < fut[k])
        out[se] = 1
        out[sf] = 0
    return (out == 1).mean()

B_by_real = {r: np.cumsum(rng_h.normal(r * dt, np.sqrt(dt), size=(n_sim_h, K)), axis=1)
             for r in theta_real}
power_eff = {r: power_from_B(eff_bound, fut_none, B_by_real[r]) for r in theta_real}

loss = np.zeros((len(theta_real), len(theta_design)))
for j, a in tqdm(enumerate(theta_design)):
    _, fut_a = group_sequential_boundaries(t, design.alpha, design.beta, a)
    for i, r in enumerate(theta_real):
        pf = power_from_B(eff_bound, fut_a, B_by_real[r])
        loss[i, j] = 100 * (power_eff[r] - pf)

fig = go.Figure(go.Heatmap(
    z=loss, x=theta_design, y=theta_real,
    colorscale="Magma",
    colorbar=dict(title="падение мощности, п.п."),
    hovertemplate="ожидаемый θ=%{x:.2f}<br>реальный θ=%{y:.2f}<br>падение мощности=%{z:.1f} п.п.<extra></extra>",
))
diag = np.linspace(1.0, 4.0, 50)
fig.add_trace(go.Scatter(x=diag, y=diag, mode="lines",
                         line=dict(color="white", width=2, dash="dash"),
                         name="θ_ожид. = θ_реал. (правильный дизайн)"))
fig.update_xaxes(title_text="ожидаемый θ (дизайн futility)")
fig.update_yaxes(title_text="реальный θ")
fig.update_layout(height=560, template="plotly_white",
                  title="Падение мощности от futility: ожидаемый θ vs реальный θ",
                  legend=dict(orientation="h", y=-0.12))
fig


71it [03:29,  2.96s/it]


#### 3.4. Почему максимумы в разных местах

Обе карты — про одну формулу: futility-граница строится под **ожидаемый** θ,

$$f_k \approx \theta_{\text{ожид}}\cdot\sqrt{t_k}+q_k,\qquad q_k=\Phi^{-1}(\Delta\beta_k)<0,$$

поэтому чем больше ожидаемый θ (правее по X), тем выше граница и тем больше траекторий она отсекает. **Правая сторона — самая агрессивная futility**, отсюда и максимумы по X.

**Экономия ASN — справа снизу.** Экономия $=\mathrm{ASN}_{\text{eff}}-\mathrm{ASN}_{\text{fut}}$. Под $H_0$ (реальный $\theta=0$) efficacy-дизайн почти не останавливается рано ($\mathrm{ASN}_{\text{eff}}\approx1$), а futility режет почти всех — разрыв максимален. С ростом реального θ efficacy тоже начинает останавливаться рано, и разрыв схлопывается.

**Потеря мощности — справа посередине.** Потеря $=\mathrm{power}_{\text{eff}}-\mathrm{power}_{\text{fut}}$, «сколько истинных эффектов futility зря остановил». Под $H_0$ терять почти нечего (мощность $=5\%$). При очень большом реальном θ эффект настолько сильный, что отклонение происходит на первом взгляде, а futility почти не срабатывает. Пик — около $\theta_{\text{MDE}}$: мощность уже высокая (есть что терять), но эффект ещё пограничный (futility может ошибочно сработать).

**Одна и та же ошибка «+1» на разном уровне θ.** Частота срабатывания futility зависит в основном от **разницы** $\theta_{\text{ожид}}-\theta_{\text{реал}}$: расстояние от истинного среднего до границы — $(\theta_{\text{ожид}}-\theta_{\text{реал}})\sqrt{t_k}+q_k$. А цена срабатывания разная: при $\theta\approx1$ остановленная траектория скорее всего и так не дошла бы до значимости (терять нечего), при $\theta\approx5$ это уже «настоящая победа» (терять дорого). Поэтому при высоком θ режем верные победы, при низком — будущие провалы.


## 4. Практические рекомендации

Ниже — правила, которые стоит принять **до запуска** теста. Числа в скобках взяты из расчётов этой лекции: раздел 2 — 19 взглядов с $t=0{,}10$ до $t=1$, OBF-подобные $\alpha$- и $\beta$-spending; раздел 3 — 5 взглядов с $t=0{,}3$ до $t=1$, $\gamma=0{,}15$.

### 4.1 Когда смотреть

1. **Первый futility-взгляд — не раньше 30–50% информации.** При малом $t$ карта CP почти плоская (раздел 1.2.2): судьбу теста решает будущее, а не накопленные данные, и при $t\to0$ CP стремится просто к априорной мощности дизайна. Ранний $Z_k$ очень шумный, поэтому стоп на нём чаще всего «убивает» тесты с реальным эффектом. Если ранний взгляд нужен (например, для контроля деградаций), закладывайте на нём почти нулевой расход $\beta$.
2. **Расписание взглядов задаётся в долях информации $t_k$, а не в календарных днях.** Если трафик пришёл не по плану, spending-функции пересчитываются на фактические $t_k$ — для этого они и нужны. Сдвигать взгляды «по ощущениям» после того, как увидели данные, нельзя.
3. **Взглядов немного.** В сквозном примере (раздел 3) хватает 5 взглядов с 30% по 100% информации: каждый дополнительный взгляд — это ещё одна возможность ошибочно остановиться, а экономия от него убывает.

### 4.2 Какой метод futility выбрать

- **$\beta$-spending — основа дизайна.** Граница $f_k$ фиксируется заранее и гарантирует, что суммарная вероятность зря остановить тест при эффекте $\theta_{\mathrm{MDE}}$ не больше заложенного $\beta$. Это единственный из трёх методов, который контролирует ошибку II рода по построению.
- **CP с $\theta_{\mathrm{MDE}}$ — понятное правило мониторинга.** Его легко объяснить бизнесу («шанс дойти до значимости меньше 15%»). В сквозном примере (раздел 3) CP-futility дал почти то же, что и $\beta$-spending: мощность 78,5% против 78,1%, средняя доля данных при $H_1$ — 75,0% против 74,3%. Но CP не гарантирует заранее, сколько мощности будет потеряно (раздел 1.2.3), — проверяйте это симуляцией.
- **PPoS — только с обоснованным prior** (см. 4.5).

### 4.3 Выбор семейства $\beta$-spending

- **$\beta(t)=\beta t^3$ (OBF-подобная) — дефолт.** Тратит бюджет поздно: $\beta(0{,}3)=0{,}2\cdot0{,}027\approx0{,}005$, $\beta(0{,}5)=0{,}025$. Ранние futility-границы низкие, тест с реальным эффектом почти не рискует остановиться на шумном начале.
- **$\beta(t)=\beta t$ (Pocock-подобная) — когда трафик дорогой и вы ожидаете, что большинство гипотез «пустые».** Тратит бюджет равномерно: $\beta(0{,}5)=0{,}10$, то есть к середине теста уже израсходована половина допустимой ошибки II рода. Пустые тесты закрываются раньше, но и тесты с реальным эффектом чаще обрываются на раннем шуме.
- Компромисс всегда один: **чем больше $\beta$ тратится рано, тем больше экономия под $H_0$ и тем выше риск потерять реальный эффект на раннем взгляде.** Сравнить семейства на своих параметрах можно в интерактиве раздела 3 (переключатель «$\beta$-spending»).

### 4.4 Порог $\gamma$ для CP и PPoS

- **Типичный диапазон — $\gamma\in[0{,}10;\,0{,}20]$**, в лекции используется $\gamma=0{,}15$.
- Выше 20% правило становится агрессивным: останавливается заметная доля тестов, которые ещё могли «добиться» значимости, мощность проседает.
- Ниже 5% экономии почти нет: стоп срабатывает только на совсем безнадёжных траекториях, которые и так почти никогда не доходят до значимости.
- Порог $\gamma$ — не бюджет ошибки II рода: $\mathrm{CP}<\gamma$ на каждом взгляде не означает, что потеря мощности не превысит $\gamma$. Итоговую потерю считайте симуляцией (как в разделах 2 и 3).

### 4.5 Какое $\theta$ подставлять в CP и какой prior брать для PPoS

- **В CP подставляйте $\theta_{\mathrm{MDE}}$** — эффект, ради которого тест проектировался. Вопрос тогда звучит так: «есть ли шанс поймать то, что нам важно».
- **Не подставляйте наблюдаемый тренд $\hat\theta_t=z_t/\sqrt{t}$.** Ранняя оценка шумная, и правило становится самосбывающимся: случайно слабое начало даёт маленький $\hat\theta_t$, маленький $\hat\theta_t$ даёт почти нулевой CP, тест останавливается — хотя истинный эффект может быть равен MDE. И наоборот, случайный ранний всплеск делает CP завышенно оптимистичным. На графике CP при $t=0{,}5$ (раздел 1.2.1) кривая «под трендом» резко отличается от кривой под $\theta_{\mathrm{MDE}}$ при тех же данных.
- **$\theta=0$** даёт слишком осторожное правило: оно почти никогда не останавливает тест.
- **PPoS со скептическим prior ($\theta_0=0$, $\tau=0{,}8$) очень консервативен.** В сквозном примере (раздел 3) он зря остановил 40,8% тестов с эффектом MDE (у $\beta$-spending — 19,4%), и мощность упала до 58,5% — примерно на 21 п.п. ниже фиксированного $z$-теста (80,0%). Если используете PPoS, берите prior с центром около $\theta_{\mathrm{MDE}}$ или построенный по истории похожих тестов, и обязательно проверяйте мощность симуляцией.

### 4.6 Binding или non-binding

- **Non-binding — дефолт.** Efficacy-граница $e_k$ считается так, будто futility нет, поэтому стоп можно отменить (например, если бизнес хочет досмотреть тест) без риска для $\alpha$. Если же стоп соблюдать, $\alpha$ получается даже ниже номинала: 0,045 в расчёте раздела 1.4.2.
- **Binding даёт более низкую $e_k$, но делает стоп обязательным.** В разделе 1.4.2 с Pocock-подобным $\beta$-spending финальная граница $e_K\approx1{,}74$ при binding против $1{,}84$ при non-binding. Если взять binding-границу и проигнорировать futility-стоп, ошибка I рода вырастет до $\alpha\approx0{,}057$ вместо 0,05.
- Futility-граница $f_k$ от выбора почти не зависит (разница не больше 0,01 в разделе 1.4.2): выбор влияет только на $e_k$.
- **Binding оправдан, только если стоп исполняется автоматически** и ни у кого нет полномочий «дать тесту ещё недельку». В продуктовых A/B-тестах так бывает редко.

### 4.7 Сколько стоит futility

- **Ошибку I рода futility не раздувает** (раздел 2): в сквозном примере (раздел 3) FPR у всех вариантов с futility не выше 5% (от 2,7% у PPoS до 5,0% у CP).
- **Мощность теряется немного.** В разделе 3: 80,0% у фиксированного $z$-теста, 78,6% у GST без futility, 78,1% у GST + futility (non-binding) — то есть около 0,5 п.п. к GST и около 2 п.п. к фиксированному тесту. Если это критично, немного увеличьте $N_{max}$ на этапе дизайна.
- **Данные под $H_1$ экономятся умеренно:** 74,3% от $N_{max}$ против 100% у фиксированного теста (около 25% экономии) и против 77,6% у GST только с efficacy (раздел 3).
- **Главная экономия — под $H_0$.** В разделе 2 при отсутствии эффекта 94,1% тестов останавливаются по futility, а в среднем используется 58,9% данных; при эффекте MDE — 19,7% futility-стопов и 68,9% данных. Futility выгоднее всего там, где много «пустых» гипотез.

### 4.8 Двусторонние тесты

- Futility превращается в **коридор вокруг нуля**: стоп, если $\lvert Z_k\rvert<f_k$; efficacy — если $\lvert Z_k\rvert>e_k$ (раздел 1.4.1). Бюджет $\beta$ считается под альтернативой с $\theta=z_{1-\alpha/2}+z_{1-\beta}$.
- Стоп по коридору означает «эффект, скорее всего, меньше MDE по модулю в обе стороны». Если для решения важен только рост метрики, честнее односторонняя постановка: явное ухудшение тогда тоже остановится по futility, причём раньше.

### 4.9 Что зафиксировать в дизайне до старта

1. $\alpha$-spending и $\beta$-spending функции (семейства и параметры).
2. Расписание взглядов в долях информации $t_k$ и $N_{max}$.
3. Метод futility ($\beta$-spending / CP / PPoS), порог $\gamma$, значение $\theta$ для CP или prior ($\theta_0$, $\tau$) для PPoS.
4. Статус границ: binding или non-binding.
5. Операционные характеристики из симуляции: $\alpha$, мощность, вероятность futility-стопа и средний объём данных под $H_0$ и $H_1$.

### 4.10 Типичные ошибки

- **Остановить тест после одного «плохого» взгляда без заранее заданного правила.** Это тот же peeking, только в обратную сторону: мощность падает непредсказуемо.
- **Пересчитывать границы по ходу теста** (менять $\gamma$, $\theta$, семейство spending-функции, глядя на данные). Гарантии по $\alpha$ и $\beta$ после этого не действуют.
- **Подставлять в CP наблюдаемый эффект** вместо $\theta_{\mathrm{MDE}}$ (см. 4.5).
- **Путать futility-стоп с «доказали, что эффекта нет».** Futility означает лишь «вряд ли докажем эффект размера MDE на оставшихся данных». Эффект меньше MDE вполне может существовать; для утверждения «эффекта нет» нужен тест эквивалентности.
- **Брать binding-границы и отменять стоп** — ошибка I рода выше номинала (см. 4.6).

### Чек-лист перед запуском

1. Первый futility-взгляд не раньше 30–50% информации, расписание задано в $t_k$.
2. Выбраны семейства $\alpha$- и $\beta$-spending; по умолчанию $\beta t^3$.
3. Для CP — $\theta=\theta_{\mathrm{MDE}}$ и $\gamma\in[0{,}10;\,0{,}20]$; для PPoS — обоснованный prior.
4. Границы non-binding, если нет автоматического исполнения стопа.
5. Симуляцией проверены $\alpha$, мощность и экономия данных под $H_0$ и $H_1$.
6. Всё перечисленное записано в дизайн теста до первого взгляда.


## 5. Итоги

### Что мы прошли

**Постановка (раздел 0).** Ранняя остановка бывает двух видов: по эффективности (efficacy — эффект уже доказан) и по бесперспективности (futility — эффект размера MDE, скорее всего, не будет доказан и к концу теста). Futility отвечает на вопрос: «стоит ли тратить оставшийся трафик?».

**Аппарат $Z$ и $B$ (раздел 1.1).** Статистика $Z_t\sim\mathcal N(\theta\sqrt{t},1)$ безразмерна, но неудобна для накопления. Величина $B(t)=\sqrt{t}\,Z_t$ — с точностью до константы накопленная сумма разниц по пользователям, то есть броуновское движение с дрейфом $\theta$: $\mathbb E[B(t)]=\theta t$, $\mathrm{Var}[B(t)]=t$, а приращения $B(t)-B(s)\sim\mathcal N(\theta(t-s),\,t-s)$ независимы от прошлого. На этом свойстве держится всё остальное. Границы переводятся между шкалами домножением на $\sqrt{t_k}$: $Z_k>e_k\iff B(t_k)>e_k\sqrt{t_k}$.

**Conditional Power (раздел 1.2).** $\mathrm{CP}(\theta)=P_\theta(Z_1>z_{1-\alpha}\mid Z_t=z_t)=1-\Phi\left(\frac{z_{1-\alpha}-\sqrt{t}\,z_t-\theta(1-t)}{\sqrt{1-t}}\right)$ — шанс дойти до значимости при текущих данных и заданном дрейфе. CP монотонно растёт по $z_t$, поэтому правило $\mathrm{CP}<\gamma$ эквивалентно $z_t<f_k^{\mathrm{CP}}$. Карта CP показала, что на ранних взглядах она почти плоская, а к концу превращается в ступеньку. Ответ сильно зависит от того, какое $\theta$ подставить: $0$, наблюдаемый тренд или $\theta_{\mathrm{MDE}}$; стандартный выбор — $\theta_{\mathrm{MDE}}$.

**Predictive Power (раздел 1.3).** $\mathrm{PPoS}=\int\mathrm{CP}(\theta)\,\pi(\theta\mid D_t)\,d\theta$ — CP, усреднённая по апостериорному распределению эффекта. При нормальном prior $\mathcal N(\theta_0,\tau^2)$ интеграл берётся в замкнутом виде. CP — частный случай PPoS, когда prior сосредоточен в одной точке ($\tau\to0$).

**Stochastic curtailment — общая рамка.** И CP, и PPoS — варианты одной идеи: на промежуточном взгляде оцениваем вероятность того, что финальный тест окажется значимым, и обрываем тест, если она ниже порога $\gamma$. Различаются они только тем, как обращаются с неизвестным $\theta$: фиксируют его или усредняют по prior. Граница при этом получается из данных, а не задаётся заранее.

**$\beta$-spending (раздел 1.4).** Зеркало $\alpha$-spending: бюджет ошибки II рода $\beta(t)$ распределяется по взглядам. Рекурсия на шкале $B$: на взгляде $k$ отрезаем правый хвост массой $\Delta\alpha_k$ под $H_0$ (получаем $e_k\sqrt{t_k}$) и левый хвост массой $\Delta\beta_k$ под $H_1$ (получаем $f_k\sqrt{t_k}$), затем сворачиваем выжившую плотность с независимым приращением и переходим к следующему взгляду. Независимость приращений делает рекурсию точной, а сумма приращений даёт ровно $\alpha$ и $\beta$. В двустороннем тесте efficacy — это $\lvert Z_k\rvert>e_k$, а futility — коридор $\lvert Z_k\rvert<f_k$.

**Binding и non-binding (раздел 1.4.2).** При non-binding $e_k$ считается так, будто futility нет, и стоп можно отменить без ущерба для $\alpha$. При binding $e_k$ ниже (с Pocock-подобным $\beta$-spending финальная $e_K\approx1{,}74$ против $1{,}84$), но стоп обязателен: если его проигнорировать, $\alpha$ вырастет примерно до 0,057. Граница $f_k$ от выбора почти не зависит.

**Влияние на ошибки (раздел 2).** Futility отсекает траектории снизу, поэтому вероятность пересечь efficacy-границу под $H_0$ может только уменьшиться — $\alpha$ не раздувается. Плата — потеря мощности: часть траекторий с реальным эффектом случайно выглядит слабо на раннем взгляде и останавливается. Взамен — экономия данных, особенно под $H_0$: в симуляции 94,1% «пустых» тестов остановились по futility, используя в среднем 58,9% данных.

**Сквозной пример (раздел 3).** На 5 взглядах (с $t=0{,}3$ до $t=1$) GST + $\beta$-spending futility сохранил $\alpha$ (4,8%), потерял около 2 п.п. мощности относительно фиксированного $z$-теста (78,1% против 80,0%) и сократил средний объём данных под $H_1$ до 74,3%. CP с $\theta_{\mathrm{MDE}}$ дал практически тот же результат (78,5% мощности, 75,0% данных). PPoS со скептическим prior оказался чрезмерно осторожным: мощность 58,5%, зря остановлено 40,8% тестов с реальным эффектом.

### Сравнение методов futility

| | CP | PPoS | $\beta$-spending |
|---|---|---|---|
| Философия | частотная, условная вероятность | байесовская, усреднение по prior | частотная, планирование дизайна |
| Предположение об эффекте | фиксированный $\theta$ (обычно $\theta_{\mathrm{MDE}}$) | prior $\mathcal N(\theta_0,\tau^2)$ на $\theta$ | $\theta_{\mathrm{MDE}}$ при расчёте границы |
| Тип границы | зависит от данных: $\mathrm{CP}<\gamma$, то есть $z_t<f_k^{\mathrm{CP}}$ | зависит от данных: $\mathrm{PPoS}<\gamma$ | фиксируется до старта: $Z_k<f_k$ |
| Что контролирует | шанс успеха на каждом взгляде | средний по prior шанс успеха на каждом взгляде | суммарную ошибку II рода $\le\beta$ |
| Результат в разделе 3 (мощность / доля данных при $H_1$) | 78,5% / 75,0% | 58,5% / 55,5% (скептический prior) | 78,1% / 74,3% |
| Когда использовать | понятное бизнесу правило мониторинга | есть обоснованный prior (история тестов) | основа дизайна с гарантией по мощности |

### Главное

1. **Futility не раздувает ошибку I рода** — она только снижает мощность (на 0,5–2 п.п. в наших примерах) и экономит данные, больше всего когда эффекта нет.
2. **Вся техника стоит на независимых приращениях $B(t)$:** из них выводятся CP, PPoS и рекурсия для $\beta$-spending.
3. **CP и PPoS — правила «стоп» по текущим данным, $\beta$-spending — граница с гарантией по $\beta$.** CP с $\theta_{\mathrm{MDE}}$ на практике близок к $\beta$-spending; CP с наблюдаемым трендом и PPoS со скептическим prior останавливают слишком много тестов с реальным эффектом.
4. **Non-binding — безопасный дефолт;** binding даёт более низкую $e_k$ только ценой обязательного стопа.
5. **Все параметры futility фиксируются до старта:** расписание взглядов, spending-функции, $\gamma$, $\theta$ или prior, binding-статус. Futility-стоп означает «эффект размера MDE вряд ли докажем», а не «эффекта нет».
